# 16 — Event Transition Failure Mechanisms

## Motivation

Notebook `15_constraint_error_trajectory_mechanisms` established that trajectory context has genuine predictive utility, but that this utility is highly non-uniform.

Key findings from Notebook 15:

- Full trajectory context improved aggregate prediction performance.
- Trajectory utility was strongly failure-family dependent.
- Workflow and grounding-state errors benefited from trajectory context.
- Constraint and tool-use errors were often harmed by the same trajectory signal.
- Compact current-to-history distance features outperformed the full 16-feature trajectory representation.
- The immediately preceding event (`t-1`) was the dominant source of useful historical information.
- Removing `t-1` from a fixed classifier caused a net loss of 34 correct predictions.
- Removing `t-2` was approximately neutral.
- Removing `t-3` caused a smaller but measurable loss.
- Role-aware distance features alone did not outperform positional distance.
- More elaborate temporal difference and ratio features did not improve substantially over direct recent-distance features.
- Learned routing and adaptive trajectory weighting showed evidence of conditional utility but did not generalize reliably under cross-fitting.

These results suggest that the main unresolved problem is representation.

Chronological position is not equivalent to functional agent state.

For example, `t-1` may represent:

- a tool call preceding an assistant response,
- an assistant message preceding a tool call,
- one tool call following another,
- one assistant message following another,
- or another interaction pattern.

The same embedding distance therefore has different meanings depending on the transition that generated it.

---

## Central hypothesis

Agent reliability failures are better modeled as failures of **functional event transitions** than as generic temporal anomalies.

Instead of asking only:

> How different is the current event from the previous event?

we ask:

> What type of agent transition occurred, and how should displacement across that transition affect the probability of each failure family?

---

## Research questions

### RQ1 — Where is `t-1` trajectory information useful?

Measure the causal utility of the previous event separately for each transition type.

Examples:

- `TOOL_CALL -> ASSISTANT`
- `ASSISTANT -> TOOL_CALL`
- `TOOL_CALL -> TOOL_CALL`
- `ASSISTANT -> ASSISTANT`

For each transition type, measure:

- support
- full-history accuracy
- accuracy with `t-1` removed
- rescues
- breaks
- net utility

---

### RQ2 — Does trajectory utility depend jointly on transition type and failure family?

Estimate:

    utility(t-1 | transition_type, true_failure_family)

This tests whether the harmful and beneficial class effects found in Notebook 15 can be localized to particular agent transitions.

---

### RQ3 — What prediction flows are caused by `t-1`?

Determine whether specific transitions systematically push predictions between particular failure families.

Examples:

    tool_use_error -> workflow_error
    constraint_error -> workflow_error
    constraint_error -> grounding_state_error

The objective is to determine whether trajectory displacement produces systematic failure-family biases under particular event structures.

---

### RQ4 — Is chronological `t-1` merely a proxy for a more meaningful relational anchor?

Compare:

- current event -> previous chronological event
- current event -> previous assistant event
- current event -> previous tool call
- previous tool call -> previous assistant event

The goal is not to add more features indiscriminately, but to identify which functional relation best explains the useful recent-history signal.

---

### RQ5 — Can a compact transition-conditioned representation outperform generic positional trajectory features?

Only after the diagnostic experiments, test a small representation combining:

- current semantic embedding
- recent distance
- event transition type
- selected relational distance

The objective is to improve representation quality before revisiting routing or Mixture-of-Experts architectures.

In [8]:
# ============================================================
# 1. Imports
# ============================================================

import numpy as np
import pandas as pd

from collections import Counter

from sentence_transformers import SentenceTransformer

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)

RANDOM_STATE = 42

CLASS_NAMES = [
    "workflow_error",
    "constraint_error",
    "tool_use_error",
    "grounding_state_error",
    "reasoning_value_error",
]

N_CLASSES = len(CLASS_NAMES)

In [9]:
# ============================================================
# 2. Load canonical trajectory exports
# ============================================================

train_events = pd.read_csv(
    "../data/processed/trajectory_events_train.csv"
)

test_events = pd.read_csv(
    "../data/processed/trajectory_events_test.csv"
)

targets = pd.read_csv(
    "../data/processed/trajectory_targets.csv"
)

train_targets = (
    targets[
        targets["split"] == "train"
    ]
    .reset_index(drop=True)
)

test_targets = (
    targets[
        targets["split"] == "test"
    ]
    .reset_index(drop=True)
)

print("Train events:", train_events.shape)
print("Test events:", test_events.shape)

print("Train targets:", train_targets.shape)
print("Test targets:", test_targets.shape)

assert len(train_targets) == 1489
assert len(test_targets) == 287

Train events: (3792, 40)
Test events: (799, 40)
Train targets: (1489, 14)
Test targets: (287, 14)


In [10]:
# ============================================================
# 3. Labels and group-safe split identifiers
# ============================================================

y_train = (
    train_targets[
        "family_label"
    ]
    .to_numpy()
    .astype(int)
)

y_test = (
    test_targets[
        "family_label"
    ]
    .to_numpy()
    .astype(int)
)

group_col = (
    "canonical_group"
    if "canonical_group" in train_targets.columns
    else "group_id"
)

groups_train = (
    train_targets[
        group_col
    ]
    .astype(str)
    .to_numpy()
)

print(
    train_targets[
        "failure_family"
    ].value_counts()
)

failure_family
workflow_error           660
constraint_error         317
grounding_state_error    244
tool_use_error           237
reasoning_value_error     31
Name: count, dtype: int64


In [11]:
# ============================================================
# 4. Reconstruct event history for each target
# ============================================================

TRAJECTORY_KEY = [
    "dataset",
    "group_id",
]


def build_history_indices(
    events_df,
    targets_df,
):

    lookup = {}

    for key, group in events_df.groupby(
        TRAJECTORY_KEY,
        sort=False,
    ):

        lookup[key] = (
            group
            .sort_values(
                "message_index"
            )
        )

    histories = []

    for _, row in targets_df.iterrows():

        key = (
            row["dataset"],
            row["group_id"],
        )

        target_index = int(
            row["message_index"]
        )

        events = lookup.get(key)

        if events is None:
            histories.append([])
            continue

        history = (
            events.loc[
                events[
                    "message_index"
                ] < target_index
            ]
            .index
            .tolist()
        )

        histories.append(
            history
        )

    return histories


train_history_indices = (
    build_history_indices(
        train_events,
        train_targets,
    )
)

history_count = np.array([
    len(x)
    for x in train_history_indices
])

np.testing.assert_array_equal(
    history_count,
    train_targets[
        "history_event_count"
    ].to_numpy(),
)

print(
    "✓ History reconstruction exact"
)

print(
    "Zero history:",
    (history_count == 0).sum()
)

✓ History reconstruction exact
Zero history: 56


In [12]:
# ============================================================
# 5. Encode target and event text
# ============================================================

if "content" in train_targets.columns:
    target_text_col = "content"
elif "current_text" in train_targets.columns:
    target_text_col = "current_text"
else:
    raise ValueError(
        "No target text column found."
    )


def make_event_text(row):

    role = str(
        row["event_role"]
    ).strip()

    content = (
        ""
        if pd.isna(row["content"])
        else str(row["content"]).strip()
    )

    prefix = f"[{role}]"

    if content.upper().startswith(
        prefix.upper()
    ):
        return content

    return (
        prefix
        + "\n"
        + content
    )


train_events[
    "event_text"
] = train_events.apply(
    make_event_text,
    axis=1,
)

encoder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

train_current_embeddings = encoder.encode(
    train_targets[
        target_text_col
    ]
    .fillna("")
    .astype(str)
    .tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

train_event_embeddings = encoder.encode(
    train_events[
        "event_text"
    ].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

X_sem_train = np.asarray(
    train_current_embeddings,
    dtype=np.float32,
)

print(
    X_sem_train.shape,
    train_event_embeddings.shape,
)

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Batches:   0%|          | 0/60 [00:00<?, ?it/s]

(1489, 384) (3792, 384)


In [13]:
# ============================================================
# 6. Build last-three historical embeddings
# ============================================================

def get_last_k_embeddings(
    history_indices,
    event_embeddings,
    k=3,
):

    n = len(history_indices)
    d = event_embeddings.shape[1]

    output = np.zeros(
        (n, k, d),
        dtype=np.float32,
    )

    for i, indices in enumerate(
        history_indices
    ):

        recent = indices[-k:]

        if not recent:
            continue

        output[
            i,
            -len(recent):,
            :
        ] = event_embeddings[
            recent
        ]

    return output


H_train_3 = get_last_k_embeddings(
    train_history_indices,
    train_event_embeddings,
    k=3,
)

In [14]:
# ============================================================
# 7. Six positional distance features
# ============================================================

def row_l1(a, b):
    return np.mean(
        np.abs(a - b),
        axis=1,
    )


def row_l2(a, b):
    return np.linalg.norm(
        a - b,
        axis=1,
    )


distance_data = {}

for pos, lag in [
    (0, 3),
    (1, 2),
    (2, 1),
]:

    h = H_train_3[
        :,
        pos,
        :
    ]

    present = (
        np.linalg.norm(
            h,
            axis=1,
        ) > 1e-8
    ).astype(float)

    distance_data[
        f"current_tminus{lag}_l1"
    ] = (
        row_l1(
            X_sem_train,
            h,
        )
        * present
    )

    distance_data[
        f"current_tminus{lag}_l2"
    ] = (
        row_l2(
            X_sem_train,
            h,
        )
        * present
    )


distance_df = pd.DataFrame(
    distance_data
)

distance_names = list(
    distance_df.columns
)

T_distance = distance_df.to_numpy(
    dtype=np.float32
)

print(
    distance_names
)

print(
    T_distance.shape
)

['current_tminus3_l1', 'current_tminus3_l2', 'current_tminus2_l1', 'current_tminus2_l2', 'current_tminus1_l1', 'current_tminus1_l2']
(1489, 6)


In [15]:
# ============================================================
# 8. Previous-event role and transition type
# ============================================================

previous_role = []
previous_event_index = []

for indices in train_history_indices:

    if len(indices) == 0:

        previous_role.append(
            "NO_HISTORY"
        )

        previous_event_index.append(
            None
        )

    else:

        idx = indices[-1]

        previous_event_index.append(
            idx
        )

        previous_role.append(
            str(
                train_events.loc[
                    idx,
                    "event_role",
                ]
            )
        )


# ------------------------------------------------------------
# Determine current role if export contains it
# ------------------------------------------------------------

if "event_role" in train_targets.columns:

    current_role = (
        train_targets[
            "event_role"
        ]
        .fillna("UNKNOWN")
        .astype(str)
        .to_numpy()
    )

else:

    # Target export may not contain explicit role.
    # Infer from target text prefix when possible.
    target_text = (
        train_targets[
            target_text_col
        ]
        .fillna("")
        .astype(str)
    )

    current_role = np.where(
        target_text.str.startswith(
            "[TOOL_CALL]"
        ),
        "TOOL_CALL",
        np.where(
            target_text.str.startswith(
                "[ASSISTANT]"
            ),
            "ASSISTANT",
            "UNKNOWN",
        ),
    )


transition_type = np.array([
    f"{prev}->{cur}"
    for prev, cur in zip(
        previous_role,
        current_role,
    )
])


transition_structure_df = pd.DataFrame({
    "previous_role":
        previous_role,

    "current_role":
        current_role,

    "transition_type":
        transition_type,

    "history_event_count":
        history_count,

    "failure_family":
        [
            CLASS_NAMES[int(y)]
            for y in y_train
        ],
})


print(
    "Unique transition types:",
    transition_structure_df[
        "transition_type"
    ].nunique()
)

display(
    transition_structure_df[
        "transition_type"
    ]
    .value_counts()
    .head(20)
)

Unique transition types: 6


transition_type
TOOL_CALL->TOOL_CALL     590
TOOL_CALL->ASSISTANT     397
ASSISTANT->ASSISTANT     226
ASSISTANT->TOOL_CALL     220
NO_HISTORY->ASSISTANT     34
NO_HISTORY->TOOL_CALL     22
Name: count, dtype: int64

In [16]:
# ============================================================
# 9. OOF full-distance vs remove-t1 intervention
# ============================================================

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

full_prob = np.zeros(
    (len(y_train), N_CLASSES)
)

remove_t1_prob = np.zeros(
    (len(y_train), N_CLASSES)
)

# T_distance order:
#
# t3_l1, t3_l2,
# t2_l1, t2_l2,
# t1_l1, t1_l2

T1_POSITIONS = [
    4,
    5,
]


for fold, (tr_idx, va_idx) in enumerate(
    cv.split(
        X_sem_train,
        y_train,
        groups=groups_train,
    ),
    start=1,
):

    X_tr = np.hstack([
        X_sem_train[
            tr_idx
        ],
        T_distance[
            tr_idx
        ],
    ])

    scaler = StandardScaler()

    X_tr_scaled = (
        scaler.fit_transform(
            X_tr
        )
    )

    model = LogisticRegression(
        C=0.01,
        max_iter=5000,
        random_state=42,
    )

    model.fit(
        X_tr_scaled,
        y_train[
            tr_idx
        ],
    )

    # --------------------------------------------------------
    # Full recent-distance representation
    # --------------------------------------------------------

    X_va_full = np.hstack([
        X_sem_train[
            va_idx
        ],
        T_distance[
            va_idx
        ],
    ])

    full_prob[
        va_idx
    ] = model.predict_proba(
        scaler.transform(
            X_va_full
        )
    )

    # --------------------------------------------------------
    # Same model; remove t-1 only
    # --------------------------------------------------------

    T_remove = (
        T_distance[
            va_idx
        ].copy()
    )

    T_remove[
        :,
        T1_POSITIONS
    ] = 0.0

    X_va_remove = np.hstack([
        X_sem_train[
            va_idx
        ],
        T_remove,
    ])

    remove_t1_prob[
        va_idx
    ] = model.predict_proba(
        scaler.transform(
            X_va_remove
        )
    )

    print(
        f"Fold {fold} complete"
    )


full_pred = (
    full_prob.argmax(
        axis=1
    )
)

remove_t1_pred = (
    remove_t1_prob.argmax(
        axis=1
    )
)

Fold 1 complete
Fold 2 complete
Fold 3 complete
Fold 4 complete
Fold 5 complete


In [17]:
# ============================================================
# 10. t-1 utility by event-transition type
# ============================================================

full_correct = (
    full_pred == y_train
)

remove_correct = (
    remove_t1_pred == y_train
)

transition_rows = []

for transition in sorted(
    np.unique(
        transition_type
    )
):

    mask = (
        transition_type
        == transition
    )

    # t-1 intervention only makes sense
    # where history exists.
    mask = (
        mask
        &
        (history_count >= 1)
    )

    n = int(
        mask.sum()
    )

    if n == 0:
        continue

    gained_when_removed = (
        mask
        &
        (~full_correct)
        &
        remove_correct
    )

    lost_when_removed = (
        mask
        &
        full_correct
        &
        (~remove_correct)
    )

    transition_rows.append({
        "transition_type":
            transition,

        "support":
            n,

        "full_accuracy":
            accuracy_score(
                y_train[mask],
                full_pred[mask],
            ),

        "without_t1_accuracy":
            accuracy_score(
                y_train[mask],
                remove_t1_pred[mask],
            ),

        "gained_when_removed":
            int(
                gained_when_removed.sum()
            ),

        "lost_when_removed":
            int(
                lost_when_removed.sum()
            ),

        # Positive means t1 helps.
        "net_t1_utility":
            int(
                lost_when_removed.sum()
                -
                gained_when_removed.sum()
            ),

        "t1_utility_rate":
            (
                lost_when_removed.sum()
                -
                gained_when_removed.sum()
            ) / n,
    })


transition_utility_df = (
    pd.DataFrame(
        transition_rows
    )
    .sort_values(
        [
            "net_t1_utility",
            "support",
        ],
        ascending=False,
    )
)

display(
    transition_utility_df.round(4)
)

,transition_type,support,full_accuracy,without_t1_accuracy,gained_when_removed,lost_when_removed,net_t1_utility,t1_utility_rate
3,TOOL_CALL->TOOL_CALL,590,0.6797,0.6475,15,34,19,0.0322
2,TOOL_CALL->ASSISTANT,397,0.4509,0.4131,13,28,15,0.0378
1,ASSISTANT->TOOL_CALL,220,0.5273,0.5091,16,20,4,0.0182
0,ASSISTANT->ASSISTANT,226,0.3274,0.3451,9,5,-4,-0.0177


In [18]:
# ============================================================
# 11. t-1 utility by transition type AND failure family
# ============================================================

rows = []

for transition in np.unique(
    transition_type
):

    for class_id, family in enumerate(
        CLASS_NAMES
    ):

        mask = (
            (transition_type == transition)
            &
            (y_train == class_id)
            &
            (history_count >= 1)
        )

        support = int(
            mask.sum()
        )

        # Avoid overinterpreting tiny cells.
        if support < 10:
            continue

        gained = (
            mask
            &
            (~full_correct)
            &
            remove_correct
        )

        lost = (
            mask
            &
            full_correct
            &
            (~remove_correct)
        )

        rows.append({
            "transition_type":
                transition,

            "failure_family":
                family,

            "support":
                support,

            "full_recall":
                (
                    full_pred[mask]
                    == class_id
                ).mean(),

            "without_t1_recall":
                (
                    remove_t1_pred[mask]
                    == class_id
                ).mean(),

            "recall_delta_t1":
                (
                    (
                        full_pred[mask]
                        == class_id
                    ).mean()
                    -
                    (
                        remove_t1_pred[mask]
                        == class_id
                    ).mean()
                ),

            "gained_when_removed":
                int(gained.sum()),

            "lost_when_removed":
                int(lost.sum()),

            "net_t1_utility":
                int(
                    lost.sum()
                    -
                    gained.sum()
                ),
        })


transition_family_utility_df = (
    pd.DataFrame(rows)
    .sort_values(
        "net_t1_utility",
        ascending=False,
    )
)

display(
    transition_family_utility_df
    .round(4)
)

,transition_type,failure_family,support,full_recall,without_t1_recall,recall_delta_t1,gained_when_removed,lost_when_removed,net_t1_utility
15,TOOL_CALL->TOOL_CALL,tool_use_error,136,0.5809,0.4118,0.1691,0,23,23
11,TOOL_CALL->ASSISTANT,grounding_state_error,97,0.4536,0.2680,0.1856,0,18,18
16,TOOL_CALL->TOOL_CALL,grounding_state_error,67,0.5970,0.4478,0.1493,0,10,10
5,ASSISTANT->TOOL_CALL,constraint_error,57,0.5614,0.4211,0.1404,0,8,8
6,ASSISTANT->TOOL_CALL,tool_use_error,36,0.5000,0.2778,0.2222,0,8,8
7,ASSISTANT->TOOL_CALL,grounding_state_error,25,0.3600,0.2000,0.1600,0,4,4
9,TOOL_CALL->ASSISTANT,constraint_error,128,0.5000,0.4688,0.0312,1,5,4
10,TOOL_CALL->ASSISTANT,tool_use_error,33,0.0909,0.0000,0.0909,0,3,3
3,ASSISTANT->ASSISTANT,grounding_state_error,42,0.0952,0.0476,0.0476,0,2,2
12,TOOL_CALL->ASSISTANT,reasoning_value_error,17,0.4118,0.2941,0.1176,0,2,2


In [19]:
# ============================================================
# 12. Prediction flows caused by t-1 information
# ============================================================

changed_by_t1 = (
    full_pred
    != remove_t1_pred
)

print(
    "Predictions affected by t-1:",
    changed_by_t1.sum()
)

t1_flow_df = pd.DataFrame({
    "transition_type":
        transition_type[
            changed_by_t1
        ],

    "true_family": [
        CLASS_NAMES[int(i)]
        for i in y_train[
            changed_by_t1
        ]
    ],

    "without_t1_prediction": [
        CLASS_NAMES[int(i)]
        for i in remove_t1_pred[
            changed_by_t1
        ]
    ],

    "with_t1_prediction": [
        CLASS_NAMES[int(i)]
        for i in full_pred[
            changed_by_t1
        ]
    ],

    "effect":
        np.where(
            (
                (~remove_correct)
                &
                full_correct
            )[
                changed_by_t1
            ],
            "rescue",
            np.where(
                (
                    remove_correct
                    &
                    (~full_correct)
                )[
                    changed_by_t1
                ],
                "break",
                "wrong_to_wrong",
            ),
        ),
})


display(
    t1_flow_df.groupby([
        "transition_type",
        "without_t1_prediction",
        "with_t1_prediction",
        "effect",
    ])
    .size()
    .reset_index(
        name="count"
    )
    .sort_values(
        "count",
        ascending=False,
    )
    .head(40)
)

Predictions affected by t-1: 181


,transition_type,without_t1_prediction,with_t1_prediction,effect,count
44,TOOL_CALL->TOOL_CALL,workflow_error,tool_use_error,rescue,23
43,TOOL_CALL->TOOL_CALL,workflow_error,tool_use_error,break,13
31,TOOL_CALL->ASSISTANT,workflow_error,grounding_state_error,rescue,13
21,ASSISTANT->TOOL_CALL,workflow_error,tool_use_error,rescue,8
15,ASSISTANT->TOOL_CALL,workflow_error,constraint_error,rescue,8
20,ASSISTANT->TOOL_CALL,workflow_error,tool_use_error,break,7
41,TOOL_CALL->TOOL_CALL,workflow_error,grounding_state_error,rescue,7
27,TOOL_CALL->ASSISTANT,workflow_error,constraint_error,break,7
28,TOOL_CALL->ASSISTANT,workflow_error,constraint_error,rescue,5
24,TOOL_CALL->ASSISTANT,constraint_error,grounding_state_error,rescue,5


In [20]:
# ============================================================
# 13b. Semantic uncertainty × transition-type utility
# ============================================================

# Confidence / entropy from the no-t1 prediction.
# This represents what the model believes using semantic +
# older-history evidence before t-1 is injected.

remove_sorted = np.sort(
    remove_t1_prob,
    axis=1,
)

semantic_like_confidence = (
    remove_sorted[:, -1]
)

semantic_like_margin = (
    remove_sorted[:, -1]
    -
    remove_sorted[:, -2]
)

semantic_like_entropy = -np.sum(
    remove_t1_prob
    *
    np.log(
        remove_t1_prob + 1e-12
    ),
    axis=1,
)


uncertainty_df = pd.DataFrame({
    "transition_type":
        transition_type,

    "confidence":
        semantic_like_confidence,

    "margin":
        semantic_like_margin,

    "entropy":
        semantic_like_entropy,

    "t1_rescue":
        (
            (~remove_correct)
            &
            full_correct
        ),

    "t1_break":
        (
            remove_correct
            &
            (~full_correct)
        ),

    "changed":
        (
            full_pred
            != remove_t1_pred
        ),
})

In [21]:
# ============================================================
# 13c. t-1 utility by transition + confidence level
# ============================================================

uncertainty_df[
    "confidence_bin"
] = pd.qcut(
    uncertainty_df[
        "confidence"
    ],
    q=3,
    labels=[
        "low",
        "medium",
        "high",
    ],
    duplicates="drop",
)


rows = []

for (
    transition,
    confidence_bin
), group in uncertainty_df.groupby(
    [
        "transition_type",
        "confidence_bin",
    ],
    observed=True,
):

    if len(group) < 15:
        continue

    rescues = int(
        group[
            "t1_rescue"
        ].sum()
    )

    breaks = int(
        group[
            "t1_break"
        ].sum()
    )

    rows.append({
        "transition_type":
            transition,

        "confidence_bin":
            confidence_bin,

        "support":
            len(group),

        "rescues":
            rescues,

        "breaks":
            breaks,

        "net_t1_utility":
            rescues - breaks,

        "utility_rate":
            (
                rescues
                -
                breaks
            ) / len(group),

        "change_rate":
            group[
                "changed"
            ].mean(),
    })


transition_confidence_utility = (
    pd.DataFrame(rows)
    .sort_values(
        "utility_rate",
        ascending=False,
    )
)

display(
    transition_confidence_utility.round(4)
)

,transition_type,confidence_bin,support,rescues,breaks,net_t1_utility,utility_rate,change_rate
7,TOOL_CALL->ASSISTANT,low,162,26,10,16,0.0988,0.2963
3,ASSISTANT->TOOL_CALL,low,71,17,10,7,0.0986,0.4930
10,TOOL_CALL->TOOL_CALL,low,135,26,13,13,0.0963,0.3407
11,TOOL_CALL->TOOL_CALL,medium,162,8,2,6,0.0370,0.0617
2,ASSISTANT->ASSISTANT,high,35,0,0,0,0.0000,0.0000
5,ASSISTANT->TOOL_CALL,high,64,0,0,0,0.0000,0.0000
6,NO_HISTORY->ASSISTANT,low,17,0,0,0,0.0000,0.0000
9,TOOL_CALL->ASSISTANT,high,91,0,0,0,0.0000,0.0000
12,TOOL_CALL->TOOL_CALL,high,293,0,0,0,0.0000,0.0000
8,TOOL_CALL->ASSISTANT,medium,144,2,3,-1,-0.0069,0.0347


In [22]:
# ============================================================
# 13d. High semantic uncertainty cases
# ============================================================

uncertainty_df[
    "entropy_bin"
] = pd.qcut(
    uncertainty_df[
        "entropy"
    ],
    q=3,
    labels=[
        "low",
        "medium",
        "high",
    ],
    duplicates="drop",
)


entropy_utility = (
    uncertainty_df
    .groupby(
        [
            "transition_type",
            "entropy_bin",
        ],
        observed=True,
    )
    .agg(
        support=(
            "changed",
            "size",
        ),

        change_rate=(
            "changed",
            "mean",
        ),

        rescues=(
            "t1_rescue",
            "sum",
        ),

        breaks=(
            "t1_break",
            "sum",
        ),
    )
    .reset_index()
)

entropy_utility[
    "net_t1_utility"
] = (
    entropy_utility[
        "rescues"
    ]
    -
    entropy_utility[
        "breaks"
    ]
)

entropy_utility[
    "utility_rate"
] = (
    entropy_utility[
        "net_t1_utility"
    ]
    /
    entropy_utility[
        "support"
    ]
)

display(
    entropy_utility
    .sort_values(
        "utility_rate",
        ascending=False,
    )
    .round(4)
)

,transition_type,entropy_bin,support,change_rate,rescues,breaks,net_t1_utility,utility_rate
14,TOOL_CALL->ASSISTANT,high,186,0.2473,25,11,14,0.0753
17,TOOL_CALL->TOOL_CALL,high,87,0.1839,8,2,6,0.0690
16,TOOL_CALL->TOOL_CALL,medium,200,0.2000,26,13,13,0.0650
4,ASSISTANT->TOOL_CALL,medium,90,0.1444,8,4,4,0.0444
13,TOOL_CALL->ASSISTANT,medium,121,0.0413,3,0,3,0.0248
10,NO_HISTORY->TOOL_CALL,medium,7,0.0000,0,0,0,0.0000
15,TOOL_CALL->TOOL_CALL,low,303,0.0000,0,0,0,0.0000
11,NO_HISTORY->TOOL_CALL,high,11,0.0000,0,0,0,0.0000
0,ASSISTANT->ASSISTANT,low,33,0.0000,0,0,0,0.0000
8,NO_HISTORY->ASSISTANT,high,18,0.0000,0,0,0,0.0000


This result is exactly the kind of interaction signal we were looking for. It says transition structure alone is too coarse, but transition structure × model uncertainty is much more informative.

The cleanest pattern is in the confidence bins. For low-confidence cases, t−1 is strongly helpful across all three tool-mediated transitions: TOOL_CALL→ASSISTANT has +16 net utility, ASSISTANT→TOOL_CALL +7, and TOOL_CALL→TOOL_CALL +13. At high confidence, t−1 contributes essentially nothing. That is a very strong gating signal.

The entropy analysis says the same thing in reverse because high entropy means uncertainty. TOOL_CALL→ASSISTANT with high entropy gets +14 net, and TOOL_CALL→TOOL_CALL with medium/high entropy gets +13 / +6. ASSISTANT→TOOL_CALL gets a smaller positive effect in its uncertain regime. Meanwhile ASSISTANT→ASSISTANT remains neutral-to-harmful even when uncertainty is high.

So the emerging mechanism is:

Use recent trajectory evidence primarily when the model is uncertain and the current event participates in a tool-mediated transition.

That is much more plausible than a hard role-only rule.

For the hallucination example you raised, this is especially important. An ASSISTANT→ASSISTANT sequence can absolutely contain a hallucination or grounding error, but these results say that the current t−1 distance feature is not a reliable way to detect it. In that transition, even high-entropy examples are net −3. That doesn't mean hallucinations are impossible there; it means this particular historical signal is not helping. Hallucination detection may require a different relation—e.g. current assistant text against the last tool result, retrieved evidence, or prior asserted facts—not merely the immediately preceding assistant embedding.

So I would not continue with the hard transition_t1_policy from Cell 13 as the main model. Keep it only as a baseline. The next experiment should directly test a tiny rule:

if tool-mediated transition
and semantic uncertainty is high:
    use t−1
else:
    suppress t−1

Use confidence/entropy thresholds selected cross-fitted, because the current bins were derived on the same OOF data.

In [23]:
# Next experiment: uncertainty-gated t−1 injection
# First create a simple eligibility mask. We should start with the three tool-mediated transitions and exclude ASSISTANT→ASSISTANT.

# ============================================================
# 18. Tool-mediated transition eligibility
# ============================================================

tool_mediated = np.isin(
    transition_type,
    [
        "TOOL_CALL->TOOL_CALL",
        "TOOL_CALL->ASSISTANT",
        "ASSISTANT->TOOL_CALL",
    ],
)

print(
    "Tool-mediated examples:",
    tool_mediated.sum(),
)

print(
    "Assistant->Assistant:",
    (
        transition_type
        == "ASSISTANT->ASSISTANT"
    ).sum(),
)

Tool-mediated examples: 1207
Assistant->Assistant: 226


In [24]:
# ============================================================
# 19. Gate inputs from model without t-1
# ============================================================

remove_sorted = np.sort(
    remove_t1_prob,
    axis=1,
)

gate_confidence = (
    remove_sorted[:, -1]
)

gate_margin = (
    remove_sorted[:, -1]
    -
    remove_sorted[:, -2]
)

gate_entropy = -np.sum(
    remove_t1_prob
    *
    np.log(
        remove_t1_prob + 1e-12
    ),
    axis=1,
)

print(
    pd.DataFrame({
        "confidence":
            gate_confidence,
        "margin":
            gate_margin,
        "entropy":
            gate_entropy,
    }).describe()
)

        confidence       margin      entropy
count  1489.000000  1489.000000  1489.000000
mean      0.643090     0.421009     0.880563
std       0.183967     0.299804     0.328741
min       0.259469     0.001126     0.162592
25%       0.492760     0.146555     0.683362
50%       0.609290     0.363389     0.924623
75%       0.792516     0.667222     1.139039
max       0.966930     0.951724     1.562557


In [28]:
# ============================================================
# Helper: classification metrics
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)

def metrics_local(pred, y_true=None):
    """
    Return the main classification metrics used throughout
    the trajectory experiments.
    """
    if y_true is None:
        y_true = y_train

    return {
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "macro_f1": f1_score(
            y_true,
            pred,
            average="macro",
            zero_division=0,
        ),
        "weighted_f1": f1_score(
            y_true,
            pred,
            average="weighted",
            zero_division=0,
        ),
    }


# Sanity check
print(
    "Full t-1:",
    metrics_local(full_pred)
)

print(
    "Without t-1:",
    metrics_local(remove_t1_pred)
)

Full t-1: {'accuracy': 0.5379449294828744, 'balanced_accuracy': 0.49832469775945765, 'macro_f1': 0.5104901329429692, 'weighted_f1': 0.5351506945358958}
Without t-1: {'accuracy': 0.5151108126259234, 'balanced_accuracy': 0.43550722847563206, 'macro_f1': 0.45483336181761536, 'weighted_f1': 0.4969282458539297}


In [29]:
# ============================================================
# 20. Diagnostic confidence-gated t-1 policy
# ============================================================

confidence_thresholds = np.linspace(
    0.30,
    0.90,
    25,
)

rows = []

for threshold in confidence_thresholds:

    use_t1 = (
        tool_mediated
        &
        (
            gate_confidence
            <= threshold
        )
    )

    pred = remove_t1_pred.copy()

    pred[
        use_t1
    ] = full_pred[
        use_t1
    ]

    m = metrics_local(pred)

    correct = (
        pred == y_train
    )

    remove_correct_local = (
        remove_t1_pred
        == y_train
    )

    rescues = (
        (~remove_correct_local)
        &
        correct
    )

    breaks = (
        remove_correct_local
        &
        (~correct)
    )

    rows.append({
        "threshold":
            threshold,

        "use_t1_count":
            int(
                use_t1.sum()
            ),

        "coverage":
            use_t1.mean(),

        **m,

        "rescues_vs_no_t1":
            int(
                rescues.sum()
            ),

        "breaks_vs_no_t1":
            int(
                breaks.sum()
            ),

        "net_vs_no_t1":
            int(
                rescues.sum()
                -
                breaks.sum()
            ),
    })


confidence_gate_df = (
    pd.DataFrame(rows)
    .sort_values(
        [
            "macro_f1",
            "accuracy",
        ],
        ascending=False,
    )
)

display(
    confidence_gate_df
    .head(15)
    .round(4)
)

,threshold,use_t1_count,coverage,accuracy,balanced_accuracy,macro_f1,weighted_f1,rescues_vs_no_t1,breaks_vs_no_t1,net_vs_no_t1
12,0.600,546,0.3667,0.5406,0.4985,0.5105,0.5368,82,44,38
13,0.625,595,0.3996,0.5406,0.4985,0.5105,0.5368,82,44,38
14,0.650,631,0.4238,0.5406,0.4985,0.5105,0.5368,82,44,38
15,0.675,669,0.4493,0.5406,0.4985,0.5105,0.5368,82,44,38
16,0.700,699,0.4694,0.5406,0.4985,0.5105,0.5368,82,44,38
17,0.725,752,0.5050,0.5406,0.4985,0.5105,0.5368,82,44,38
18,0.750,791,0.5312,0.5406,0.4985,0.5105,0.5368,82,44,38
19,0.775,830,0.5574,0.5406,0.4985,0.5105,0.5368,82,44,38
20,0.800,870,0.5843,0.5406,0.4985,0.5105,0.5368,82,44,38
21,0.825,892,0.5991,0.5406,0.4985,0.5105,0.5368,82,44,38


In [30]:
# ============================================================
# 21. Diagnostic entropy-gated t-1 policy
# ============================================================

entropy_thresholds = np.linspace(
    np.quantile(
        gate_entropy,
        0.20,
    ),
    np.quantile(
        gate_entropy,
        0.90,
    ),
    25,
)

rows = []

for threshold in entropy_thresholds:

    use_t1 = (
        tool_mediated
        &
        (
            gate_entropy
            >= threshold
        )
    )

    pred = remove_t1_pred.copy()

    pred[
        use_t1
    ] = full_pred[
        use_t1
    ]

    m = metrics_local(pred)

    correct = (
        pred == y_train
    )

    baseline_correct = (
        remove_t1_pred == y_train
    )

    rescues = (
        (~baseline_correct)
        &
        correct
    )

    breaks = (
        baseline_correct
        &
        (~correct)
    )

    rows.append({
        "threshold":
            threshold,

        "use_t1_count":
            int(
                use_t1.sum()
            ),

        "coverage":
            use_t1.mean(),

        **m,

        "rescues_vs_no_t1":
            int(
                rescues.sum()
            ),

        "breaks_vs_no_t1":
            int(
                breaks.sum()
            ),

        "net_vs_no_t1":
            int(
                rescues.sum()
                -
                breaks.sum()
            ),
    })


entropy_gate_df = (
    pd.DataFrame(rows)
    .sort_values(
        [
            "macro_f1",
            "accuracy",
        ],
        ascending=False,
    )
)

display(
    entropy_gate_df
    .head(15)
    .round(4)
)

,threshold,use_t1_count,coverage,accuracy,balanced_accuracy,macro_f1,weighted_f1,rescues_vs_no_t1,breaks_vs_no_t1,net_vs_no_t1
9,0.8337,705,0.4735,0.5426,0.4994,0.5116,0.5385,82,41,41
7,0.7756,766,0.5144,0.5420,0.4991,0.5112,0.5379,82,42,40
8,0.8046,736,0.4943,0.5420,0.4991,0.5112,0.5379,82,42,40
0,0.5722,918,0.6165,0.5406,0.4985,0.5105,0.5368,82,44,38
1,0.6013,906,0.6085,0.5406,0.4985,0.5105,0.5368,82,44,38
2,0.6303,885,0.5944,0.5406,0.4985,0.5105,0.5368,82,44,38
3,0.6594,872,0.5856,0.5406,0.4985,0.5105,0.5368,82,44,38
4,0.6884,857,0.5756,0.5406,0.4985,0.5105,0.5368,82,44,38
5,0.7175,839,0.5635,0.5406,0.4985,0.5105,0.5368,82,44,38
6,0.7465,804,0.5400,0.5406,0.4985,0.5105,0.5368,82,44,38


In [31]:
# ============================================================
# 22. Cross-fitted uncertainty gate
# ============================================================

policy_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=123,
)

crossfit_uncertainty_pred = np.empty_like(
    y_train
)

threshold_rows = []


for fold, (tr_idx, va_idx) in enumerate(
    policy_cv.split(
        X_sem_train,
        y_train,
        groups=groups_train,
    ),
    start=1,
):

    # ---------------------------------------------
    # Select threshold on TRAIN portion only
    # ---------------------------------------------

    candidates = []

    for threshold in confidence_thresholds:

        train_use = (
            tool_mediated[tr_idx]
            &
            (
                gate_confidence[tr_idx]
                <= threshold
            )
        )

        train_pred = (
            remove_t1_pred[
                tr_idx
            ].copy()
        )

        train_pred[
            train_use
        ] = full_pred[
            tr_idx
        ][
            train_use
        ]

        candidates.append({
            "threshold":
                threshold,

            "macro_f1":
                f1_score(
                    y_train[tr_idx],
                    train_pred,
                    average="macro",
                    zero_division=0,
                ),

            "accuracy":
                accuracy_score(
                    y_train[tr_idx],
                    train_pred,
                ),
        })


    candidate_df = pd.DataFrame(
        candidates
    )

    best = (
        candidate_df
        .sort_values(
            [
                "macro_f1",
                "accuracy",
            ],
            ascending=False,
        )
        .iloc[0]
    )

    threshold = float(
        best[
            "threshold"
        ]
    )

    # ---------------------------------------------
    # Apply to held-out fold
    # ---------------------------------------------

    eval_use = (
        tool_mediated[
            va_idx
        ]
        &
        (
            gate_confidence[
                va_idx
            ]
            <= threshold
        )
    )

    crossfit_uncertainty_pred[
        va_idx
    ] = remove_t1_pred[
        va_idx
    ]

    selected_idx = (
        va_idx[
            eval_use
        ]
    )

    crossfit_uncertainty_pred[
        selected_idx
    ] = full_pred[
        selected_idx
    ]

    threshold_rows.append({
        "fold":
            fold,

        "selected_threshold":
            threshold,

        "train_macro_f1":
            best[
                "macro_f1"
            ],

        "eval_use_t1":
            int(
                eval_use.sum()
            ),
    })


threshold_policy_df = pd.DataFrame(
    threshold_rows
)

In [32]:
print(
    "No t1:",
    metrics_local(
        remove_t1_pred
    )
)

print(
    "Full t1:",
    metrics_local(
        full_pred
    )
)

print(
    "Cross-fitted uncertainty gate:",
    metrics_local(
        crossfit_uncertainty_pred
    )
)

display(
    threshold_policy_df
)

No t1: {'accuracy': 0.5151108126259234, 'balanced_accuracy': 0.43550722847563206, 'macro_f1': 0.45483336181761536, 'weighted_f1': 0.4969282458539297}
Full t1: {'accuracy': 0.5379449294828744, 'balanced_accuracy': 0.49832469775945765, 'macro_f1': 0.5104901329429692, 'weighted_f1': 0.5351506945358958}
Cross-fitted uncertainty gate: {'accuracy': 0.5406312961719275, 'balanced_accuracy': 0.4985035353153443, 'macro_f1': 0.5105232564309933, 'weighted_f1': 0.5367767448588544}


,fold,selected_threshold,train_macro_f1,eval_use_t1
0,1,0.6,0.515058,101
1,2,0.6,0.499731,115
2,3,0.6,0.527090,97
3,4,0.6,0.517439,112
4,5,0.6,0.492483,121


These results sharpen the mechanism substantially. The uncertainty gate is doing something real, but it also tells us not to spend much more time tuning a scalar confidence threshold.

What this experiment establishes

The strongest baseline comparison is:

Model	Accuracy	Balanced acc.	Macro-F1
No t−1	0.5151	0.4355	0.4548
Full t−1	0.5379	0.4983	0.5105
Cross-fitted uncertainty gate	0.5406	0.4985	0.5105

So t−1 is clearly valuable. Removing it costs 2.28 accuracy points and, more importantly, about 5.6 macro-F1 points.

But always using t−1 is not optimal. The cross-fitted gate recovers another +0.27 accuracy points without sacrificing macro-F1.

More interestingly, every fold independently selected:

threshold = 0.60

That stability is important. This isn't a threshold chosen because of one lucky fold.

But look at the threshold plateau

Your first table contains an even more revealing result.

From approximately:

confidence threshold = 0.60
...
confidence threshold = 0.90

you obtain exactly:

accuracy       0.5406
balanced_acc   0.4985
macro_f1       0.5105
weighted_f1    0.5368
rescues        82
breaks         44
net            +38

even though coverage moves enormously:

36.7% → 66.3%

That means hundreds of additional examples are being routed to the t−1 model without changing their predicted class.

So uncertainty is mostly identifying a region where trajectory information has the opportunity to matter, rather than continuously controlling its usefulness.

That's an important distinction for the eventual architecture.

More importantly: your hallucination question

You were right to challenge the simple transition policy.

We previously had:

transition_t1_policy = {
    "TOOL_CALL->TOOL_CALL": True,
    "TOOL_CALL->ASSISTANT": True,
    "ASSISTANT->TOOL_CALL": True,
    "ASSISTANT->ASSISTANT": False,
    "NO_HISTORY->ASSISTANT": False,
    "NO_HISTORY->TOOL_CALL": False,
}

The current evidence says this is too coarse.

In particular, TOOL_CALL->ASSISTANT is precisely where a grounding/hallucination-type failure can occur.

You already found:

TOOL_CALL -> ASSISTANT
grounding_state_error


support:              97
full recall:          0.4536
without t1 recall:    0.2680
Δ recall:            +0.1856
net t1 utility:      +18

That is one of the strongest mechanisms in the notebook.

So the role transition is not saying:

"Use trajectory for tool-mediated examples."

It's saying something closer to:

The meaning of displacement depends on the event relation in which that displacement occurred.

For:

TOOL_CALL → ASSISTANT

large semantic displacement might indicate that the assistant response is poorly grounded in what just happened.

For:

TOOL_CALL → TOOL_CALL

displacement may instead indicate tool-selection/argument/workflow changes.

For:

ASSISTANT → TOOL_CALL

it may indicate whether the chosen action follows coherently from the preceding reasoning.

And:

ASSISTANT → ASSISTANT

isn't necessarily "trajectory useless." Your aggregate result is merely:

net t1 utility = -4 / 226

It could still contain a small subgroup where previous-assistant displacement predicts hallucination/reasoning failures.

This leads to a better experiment than another threshold search.

Next experiment: does distance mean different things by transition?

Instead of asking:

Should we use t−1?

ask:

What does t−1 displacement predict conditional on the transition type?

We should first test this diagnostically without training another complicated model.

In [38]:
# ============================================================
# Safe variable discovery
# ============================================================

import numpy as np
import pandas as pd

items = list(globals().items())

print("=== DATAFRAMES ===")

for name, obj in items:
    if isinstance(obj, pd.DataFrame) and len(obj) == 1489:
        print(
            f"{name:40s}",
            obj.shape,
            list(obj.columns)[:25]
        )

print("\n=== 2D NUMPY ARRAYS ===")

for name, obj in items:
    if isinstance(obj, np.ndarray):
        if obj.ndim == 2 and obj.shape[0] == 1489:
            print(f"{name:40s}", obj.shape)

# print("\n=== FEATURE-NAME LISTS ===")

# for name, obj in items:
#     if isinstance(obj, (list, tuple)):
#         if any(
#             "tminus" in str(x).lower()
#             for x in obj
#         ):
#             print(
#                 f"{name:40s}",
#                 f"len={len(obj)}",
#                 list(obj)[:25]
#             )
        
# ============================================================
# Find the 6-column positional-distance representation
# ============================================================

candidate_6d = []

for name, obj in list(globals().items()):

    if isinstance(obj, pd.DataFrame):
        if obj.shape == (1489, 6):
            candidate_6d.append({
                "name": name,
                "type": "DataFrame",
                "shape": obj.shape,
                "columns": list(obj.columns),
            })

    elif isinstance(obj, np.ndarray):
        if obj.shape == (1489, 6):
            candidate_6d.append({
                "name": name,
                "type": "ndarray",
                "shape": obj.shape,
                "columns": None,
            })

print("6-dimensional candidates:", len(candidate_6d))

for item in candidate_6d:
    print("\n", item)

=== DATAFRAMES ===
train_targets                            (1489, 14) ['dataset', 'group_id', 'canonical_group', 'split', 'trajectory_index', 'message_index', 'event_position', 'history_event_count', 'has_history', 'event_role', 'primary_tool', 'content', 'family_label', 'failure_family']
transition_structure_df                  (1489, 5) ['previous_role', 'current_role', 'transition_type', 'history_event_count', 'failure_family']
distance_df                              (1489, 6) ['current_tminus3_l1', 'current_tminus3_l2', 'current_tminus2_l1', 'current_tminus2_l2', 'current_tminus1_l1', 'current_tminus1_l2']
uncertainty_df                           (1489, 9) ['transition_type', 'confidence', 'margin', 'entropy', 't1_rescue', 't1_break', 'changed', 'confidence_bin', 'entropy_bin']

=== 2D NUMPY ARRAYS ===
full_prob                                (1489, 5)
remove_t1_prob                           (1489, 5)
train_current_embeddings                 (1489, 384)
X_sem_train              

In [39]:
# ============================================================
# 23. Transition × failure-family displacement mechanisms
# ============================================================

import numpy as np
import pandas as pd

t1_l1 = distance_df[
    "current_tminus1_l1"
].to_numpy()

t1_l2 = distance_df[
    "current_tminus1_l2"
].to_numpy()

mechanism_df = pd.DataFrame({
    "transition_type":
        transition_structure_df["transition_type"].to_numpy(),

    "true_family":
        train_targets["failure_family"].to_numpy(),

    "t1_l1":
        t1_l1,

    "t1_l2":
        t1_l2,

    "confidence":
        uncertainty_df["confidence"].to_numpy(),

    "margin":
        uncertainty_df["margin"].to_numpy(),

    "entropy":
        uncertainty_df["entropy"].to_numpy(),

    "full_prediction": [
        CLASS_NAMES[int(i)]
        for i in full_pred
    ],

    "no_t1_prediction": [
        CLASS_NAMES[int(i)]
        for i in remove_t1_pred
    ],
})

mechanism_df["full_correct"] = (
    mechanism_df["full_prediction"]
    == mechanism_df["true_family"]
)

mechanism_df["no_t1_correct"] = (
    mechanism_df["no_t1_prediction"]
    == mechanism_df["true_family"]
)

mechanism_df["t1_effect"] = np.select(
    [
        (~mechanism_df["no_t1_correct"])
        & mechanism_df["full_correct"],

        mechanism_df["no_t1_correct"]
        & (~mechanism_df["full_correct"]),

        (~mechanism_df["no_t1_correct"])
        & (~mechanism_df["full_correct"])
        & (
            mechanism_df["full_prediction"]
            != mechanism_df["no_t1_prediction"]
        ),
    ],
    [
        "rescue",
        "break",
        "wrong_to_wrong",
    ],
    default="unchanged",
)

print(mechanism_df["t1_effect"].value_counts())
display(mechanism_df.head())

t1_effect
unchanged         1308
rescue              87
break               53
wrong_to_wrong      41
Name: count, dtype: int64


,transition_type,true_family,t1_l1,t1_l2,confidence,margin,entropy,full_prediction,no_t1_prediction,full_correct,no_t1_correct,t1_effect
0,TOOL_CALL->ASSISTANT,constraint_error,0.039915,0.973616,0.923623,0.887817,0.344221,grounding_state_error,grounding_state_error,False,False,unchanged
1,ASSISTANT->TOOL_CALL,tool_use_error,0.043941,1.090362,0.857731,0.793774,0.552040,workflow_error,workflow_error,False,False,unchanged
2,TOOL_CALL->TOOL_CALL,workflow_error,0.009315,0.230429,0.407353,0.112967,1.225742,constraint_error,constraint_error,False,False,unchanged
3,TOOL_CALL->ASSISTANT,constraint_error,0.041609,1.002589,0.874368,0.813521,0.518790,constraint_error,constraint_error,True,True,unchanged
4,TOOL_CALL->ASSISTANT,constraint_error,0.032973,0.821834,0.678562,0.498143,0.920595,constraint_error,constraint_error,True,True,unchanged


In [40]:
# ============================================================
# 24. t-1 geometry for rescues vs breaks
# ============================================================

affected = mechanism_df[
    mechanism_df["t1_effect"].isin(
        ["rescue", "break"]
    )
].copy()

effect_summary = (
    affected
    .groupby([
        "transition_type",
        "t1_effect",
    ])
    .agg(
        count=("t1_effect", "size"),

        mean_l1=("t1_l1", "mean"),
        median_l1=("t1_l1", "median"),

        mean_l2=("t1_l2", "mean"),
        median_l2=("t1_l2", "median"),

        mean_confidence=("confidence", "mean"),
        mean_margin=("margin", "mean"),
        mean_entropy=("entropy", "mean"),
    )
    .reset_index()
)

display(
    effect_summary.sort_values(
        ["transition_type", "t1_effect"]
    )
)

,transition_type,t1_effect,count,mean_l1,median_l1,mean_l2,median_l2,mean_confidence,mean_margin,mean_entropy
0,ASSISTANT->ASSISTANT,break,9,0.032747,0.035823,0.808951,0.873398,0.451203,0.140182,1.183739
1,ASSISTANT->ASSISTANT,rescue,5,0.029982,0.030083,0.744417,0.746661,0.406762,0.053824,1.139704
2,ASSISTANT->TOOL_CALL,break,16,0.050498,0.051498,1.247436,1.266078,0.469752,0.130064,1.139218
3,ASSISTANT->TOOL_CALL,rescue,20,0.046985,0.046751,1.156007,1.154094,0.472573,0.107387,1.094644
4,TOOL_CALL->ASSISTANT,break,13,0.046669,0.047436,1.149622,1.143393,0.454262,0.103803,1.148326
5,TOOL_CALL->ASSISTANT,rescue,28,0.047741,0.047130,1.174314,1.161402,0.422627,0.098840,1.216984
6,TOOL_CALL->TOOL_CALL,break,15,0.033139,0.034352,0.821899,0.833563,0.493613,0.076693,0.960537
7,TOOL_CALL->TOOL_CALL,rescue,34,0.037482,0.038673,0.925871,0.958983,0.489338,0.087383,0.995237


In [41]:
# ============================================================
# 25. Rescue/break mechanism by transition × failure family
# ============================================================

effect_family = (
    affected
    .groupby([
        "transition_type",
        "true_family",
        "t1_effect",
    ])
    .agg(
        count=("t1_effect", "size"),

        mean_l1=("t1_l1", "mean"),
        median_l1=("t1_l1", "median"),

        mean_l2=("t1_l2", "mean"),
        median_l2=("t1_l2", "median"),

        mean_confidence=("confidence", "mean"),
        mean_entropy=("entropy", "mean"),
    )
    .reset_index()
)

display(
    effect_family[
        effect_family["count"] >= 3
    ]
    .sort_values([
        "transition_type",
        "true_family",
        "t1_effect",
    ])
)

,transition_type,true_family,t1_effect,count,mean_l1,median_l1,mean_l2,median_l2,mean_confidence,mean_entropy
0,ASSISTANT->ASSISTANT,constraint_error,break,3,0.020476,0.019124,0.502869,0.465867,0.345914,1.360943
1,ASSISTANT->ASSISTANT,constraint_error,rescue,3,0.029686,0.030070,0.733962,0.737513,0.432381,1.077526
3,ASSISTANT->ASSISTANT,workflow_error,break,6,0.038883,0.037287,0.961991,0.931869,0.503847,1.095136
4,ASSISTANT->TOOL_CALL,constraint_error,rescue,8,0.045876,0.045929,1.133647,1.124207,0.485593,1.020859
5,ASSISTANT->TOOL_CALL,grounding_state_error,rescue,4,0.046589,0.046705,1.141917,1.143285,0.460495,1.110411
6,ASSISTANT->TOOL_CALL,tool_use_error,rescue,8,0.048292,0.052282,1.185411,1.285775,0.465592,1.160546
7,ASSISTANT->TOOL_CALL,workflow_error,break,16,0.050498,0.051498,1.247436,1.266078,0.469752,1.139218
9,TOOL_CALL->ASSISTANT,constraint_error,rescue,5,0.047837,0.050859,1.170910,1.232721,0.476774,1.113157
10,TOOL_CALL->ASSISTANT,grounding_state_error,rescue,18,0.048489,0.046753,1.193692,1.160044,0.417905,1.228142
12,TOOL_CALL->ASSISTANT,tool_use_error,rescue,3,0.043466,0.044183,1.064276,1.078130,0.383327,1.212076


In [42]:
# ============================================================
# 26. TOOL_CALL -> ASSISTANT grounding mechanism
# ============================================================

from sklearn.metrics import roc_auc_score

tc_assistant = mechanism_df[
    mechanism_df["transition_type"]
    == "TOOL_CALL->ASSISTANT"
].copy()

tc_assistant["is_grounding_error"] = (
    tc_assistant["true_family"]
    == "grounding_state_error"
)

display(
    tc_assistant
    .groupby("is_grounding_error")
    .agg(
        support=("is_grounding_error", "size"),
        mean_l1=("t1_l1", "mean"),
        median_l1=("t1_l1", "median"),
        mean_l2=("t1_l2", "mean"),
        median_l2=("t1_l2", "median"),
        mean_confidence=("confidence", "mean"),
        mean_entropy=("entropy", "mean"),
    )
)

y_ground = (
    tc_assistant["is_grounding_error"]
    .astype(int)
    .to_numpy()
)

auc_l1 = roc_auc_score(
    y_ground,
    tc_assistant["t1_l1"]
)

auc_l2 = roc_auc_score(
    y_ground,
    tc_assistant["t1_l2"]
)

print(
    "L1 raw AUC:",
    auc_l1,
    "| direction-free separation:",
    max(auc_l1, 1 - auc_l1),
)

print(
    "L2 raw AUC:",
    auc_l2,
    "| direction-free separation:",
    max(auc_l2, 1 - auc_l2),
)

,support,mean_l1,median_l1,mean_l2,median_l2,mean_confidence,mean_entropy
is_grounding_error,,,,,,,
False,300,0.044595,0.045732,1.097902,1.119049,0.600466,0.983302
True,97,0.045245,0.045739,1.112939,1.121020,0.554391,1.059311


L1 raw AUC: 0.5223024054982818 | direction-free separation: 0.5223024054982818
L2 raw AUC: 0.5192439862542955 | direction-free separation: 0.5192439862542955


In [43]:
# ============================================================
# 27. Semantic confusion × transition × t-1 utility
# ============================================================

analysis_df = mechanism_df.copy()

# We specifically care about what the semantic/no-t1 model
# believed BEFORE t-1 information changed the decision.
analysis_df["semantic_prediction"] = (
    analysis_df["no_t1_prediction"]
)

analysis_df["trajectory_prediction"] = (
    analysis_df["full_prediction"]
)

affected = analysis_df[
    analysis_df["t1_effect"] != "unchanged"
].copy()

pair_summary = (
    affected
    .groupby([
        "transition_type",
        "semantic_prediction",
        "trajectory_prediction",
        "t1_effect",
    ])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(pair_summary.head(50))

,transition_type,semantic_prediction,trajectory_prediction,t1_effect,count
44,TOOL_CALL->TOOL_CALL,workflow_error,tool_use_error,rescue,23
43,TOOL_CALL->TOOL_CALL,workflow_error,tool_use_error,break,13
31,TOOL_CALL->ASSISTANT,workflow_error,grounding_state_error,rescue,13
21,ASSISTANT->TOOL_CALL,workflow_error,tool_use_error,rescue,8
15,ASSISTANT->TOOL_CALL,workflow_error,constraint_error,rescue,8
20,ASSISTANT->TOOL_CALL,workflow_error,tool_use_error,break,7
41,TOOL_CALL->TOOL_CALL,workflow_error,grounding_state_error,rescue,7
27,TOOL_CALL->ASSISTANT,workflow_error,constraint_error,break,7
28,TOOL_CALL->ASSISTANT,workflow_error,constraint_error,rescue,5
24,TOOL_CALL->ASSISTANT,constraint_error,grounding_state_error,rescue,5


In [44]:
# ============================================================
# 28. Utility of each semantic -> trajectory prediction move
# ============================================================

changed = analysis_df[
    analysis_df["semantic_prediction"]
    != analysis_df["trajectory_prediction"]
].copy()

move_utility = (
    changed
    .groupby([
        "transition_type",
        "semantic_prediction",
        "trajectory_prediction",
    ])
    .agg(
        support=("t1_effect", "size"),

        rescues=(
            "t1_effect",
            lambda x: (x == "rescue").sum()
        ),

        breaks=(
            "t1_effect",
            lambda x: (x == "break").sum()
        ),

        wrong_to_wrong=(
            "t1_effect",
            lambda x: (x == "wrong_to_wrong").sum()
        ),
    )
    .reset_index()
)

move_utility["net"] = (
    move_utility["rescues"]
    - move_utility["breaks"]
)

move_utility["utility_rate"] = (
    move_utility["net"]
    / move_utility["support"]
)

move_utility["rescue_rate"] = (
    move_utility["rescues"]
    / move_utility["support"]
)

display(
    move_utility.sort_values(
        ["net", "support"],
        ascending=[False, False]
    ).head(50)
)

,transition_type,semantic_prediction,trajectory_prediction,support,rescues,breaks,wrong_to_wrong,net,utility_rate,rescue_rate
20,TOOL_CALL->TOOL_CALL,workflow_error,tool_use_error,39,23,13,3,10,0.256410,0.589744
14,TOOL_CALL->ASSISTANT,workflow_error,grounding_state_error,19,13,3,3,10,0.526316,0.684211
19,TOOL_CALL->TOOL_CALL,workflow_error,grounding_state_error,12,7,2,3,5,0.416667,0.583333
8,ASSISTANT->TOOL_CALL,workflow_error,constraint_error,15,8,4,3,4,0.266667,0.533333
11,TOOL_CALL->ASSISTANT,constraint_error,grounding_state_error,8,5,1,2,4,0.500000,0.625000
17,TOOL_CALL->TOOL_CALL,constraint_error,grounding_state_error,3,3,0,0,3,1.000000,1.000000
10,ASSISTANT->TOOL_CALL,workflow_error,tool_use_error,18,8,7,3,1,0.055556,0.444444
3,ASSISTANT->ASSISTANT,workflow_error,constraint_error,9,3,2,4,1,0.111111,0.333333
16,TOOL_CALL->ASSISTANT,workflow_error,tool_use_error,9,3,2,4,1,0.111111,0.333333
18,TOOL_CALL->TOOL_CALL,workflow_error,constraint_error,2,1,0,1,1,0.500000,0.500000


In [45]:
# ============================================================
# 29. Where does trajectory move semantic workflow predictions?
# ============================================================

workflow_moves = changed[
    changed["semantic_prediction"]
    == "workflow_error"
].copy()

workflow_move_summary = (
    workflow_moves
    .groupby([
        "transition_type",
        "trajectory_prediction",
    ])
    .agg(
        support=("t1_effect", "size"),

        rescues=(
            "t1_effect",
            lambda x: (x == "rescue").sum()
        ),

        breaks=(
            "t1_effect",
            lambda x: (x == "break").sum()
        ),

        wrong_to_wrong=(
            "t1_effect",
            lambda x: (x == "wrong_to_wrong").sum()
        ),
    )
    .reset_index()
)

workflow_move_summary["net"] = (
    workflow_move_summary["rescues"]
    - workflow_move_summary["breaks"]
)

display(
    workflow_move_summary.sort_values(
        "net",
        ascending=False
    )
)

,transition_type,trajectory_prediction,support,rescues,breaks,wrong_to_wrong,net
7,TOOL_CALL->ASSISTANT,grounding_state_error,19,13,3,3,10
12,TOOL_CALL->TOOL_CALL,tool_use_error,39,23,13,3,10
11,TOOL_CALL->TOOL_CALL,grounding_state_error,12,7,2,3,5
3,ASSISTANT->TOOL_CALL,constraint_error,15,8,4,3,4
0,ASSISTANT->ASSISTANT,constraint_error,9,3,2,4,1
5,ASSISTANT->TOOL_CALL,tool_use_error,18,8,7,3,1
8,TOOL_CALL->ASSISTANT,reasoning_value_error,1,1,0,0,1
9,TOOL_CALL->ASSISTANT,tool_use_error,9,3,2,4,1
10,TOOL_CALL->TOOL_CALL,constraint_error,2,1,0,1,1
2,ASSISTANT->ASSISTANT,tool_use_error,1,0,1,0,-1


In [48]:
# ============================================================
# 30A. Verify whether t-1 ever moves predictions INTO workflow
# ============================================================

print(
    "Changed predictions:",
    len(changed)
)

print(
    "Moves into workflow:",
    (
        changed["trajectory_prediction"]
        == "workflow_error"
    ).sum()
)

print(
    "Moves out of workflow:",
    (
        changed["semantic_prediction"]
        == "workflow_error"
    ).sum()
)

print("\nTrajectory predictions among changed rows:")
print(
    changed[
        "trajectory_prediction"
    ].value_counts()
)

print("\nSemantic predictions among changed rows:")
print(
    changed[
        "semantic_prediction"
    ].value_counts()
)

Changed predictions: 181
Moves into workflow: 0
Moves out of workflow: 159

Trajectory predictions among changed rows:
trajectory_prediction
grounding_state_error    69
tool_use_error           68
constraint_error         42
reasoning_value_error     2
Name: count, dtype: int64

Semantic predictions among changed rows:
semantic_prediction
workflow_error           159
constraint_error          20
reasoning_value_error      1
tool_use_error             1
Name: count, dtype: int64


In [49]:
# ============================================================
# 31. Workflow refinement opportunity
# ============================================================

workflow_mask = (
    mechanism_df["no_t1_prediction"]
    == "workflow_error"
)

workflow_cases = (
    mechanism_df[
        workflow_mask
    ]
    .copy()
)

print(
    "No-t1 workflow predictions:",
    len(workflow_cases)
)

print("\nTrue families inside semantic workflow predictions:")

display(
    workflow_cases[
        "true_family"
    ]
    .value_counts()
    .rename("count")
    .to_frame()
)

print("\nTrue-family distribution by transition:")

display(
    pd.crosstab(
        workflow_cases[
            "transition_type"
        ],
        workflow_cases[
            "true_family"
        ],
        normalize="index",
    ).round(3)
)

No-t1 workflow predictions: 873

True families inside semantic workflow predictions:


,count
true_family,
workflow_error,483
constraint_error,146
tool_use_error,131
grounding_state_error,107
reasoning_value_error,6



True-family distribution by transition:


true_family,constraint_error,grounding_state_error,reasoning_value_error,tool_use_error,workflow_error
transition_type,,,,,
ASSISTANT->ASSISTANT,0.345,0.173,0.000,0.064,0.418
ASSISTANT->TOOL_CALL,0.216,0.108,0.007,0.173,0.496
NO_HISTORY->ASSISTANT,0.294,0.000,0.000,0.118,0.588
NO_HISTORY->TOOL_CALL,0.000,0.300,0.100,0.300,0.300
TOOL_CALL->ASSISTANT,0.268,0.205,0.021,0.121,0.384
TOOL_CALL->TOOL_CALL,0.054,0.076,0.000,0.177,0.693


In [50]:
# ============================================================
# 32. Workflow refinement: available vs captured corrections
# ============================================================

workflow_cases[
    "semantic_correct"
] = (
    workflow_cases[
        "no_t1_prediction"
    ]
    ==
    workflow_cases[
        "true_family"
    ]
)

workflow_cases[
    "trajectory_correct"
] = (
    workflow_cases[
        "full_prediction"
    ]
    ==
    workflow_cases[
        "true_family"
    ]
)

workflow_cases[
    "could_refine"
] = (
    ~workflow_cases[
        "semantic_correct"
    ]
)

workflow_cases[
    "successfully_refined"
] = (
    workflow_cases[
        "could_refine"
    ]
    &
    workflow_cases[
        "trajectory_correct"
    ]
)

summary = (
    workflow_cases
    .groupby(
        "transition_type"
    )
    .agg(
        support=(
            "true_family",
            "size",
        ),

        semantic_correct=(
            "semantic_correct",
            "sum",
        ),

        refinement_opportunities=(
            "could_refine",
            "sum",
        ),

        successful_refinements=(
            "successfully_refined",
            "sum",
        ),
    )
)

summary[
    "refinement_capture_rate"
] = (
    summary[
        "successful_refinements"
    ]
    /
    summary[
        "refinement_opportunities"
    ].replace(0, np.nan)
)

display(
    summary.round(4)
)

,support,semantic_correct,refinement_opportunities,successful_refinements,refinement_capture_rate
transition_type,,,,,
ASSISTANT->ASSISTANT,110,46,64,4,0.0625
ASSISTANT->TOOL_CALL,139,69,70,19,0.2714
NO_HISTORY->ASSISTANT,17,10,7,0,0.0000
NO_HISTORY->TOOL_CALL,10,3,7,0,0.0000
TOOL_CALL->ASSISTANT,190,73,117,22,0.1880
TOOL_CALL->TOOL_CALL,407,282,125,31,0.2480


In [51]:
# ============================================================
# 33A. TOOL_CALL -> TOOL_CALL
#      Semantic workflow prediction:
#      is it actually tool_use_error?
# ============================================================

subset_tc_tc = workflow_cases[
    workflow_cases[
        "transition_type"
    ]
    == "TOOL_CALL->TOOL_CALL"
].copy()

subset_tc_tc[
    "target_tool_use"
] = (
    subset_tc_tc[
        "true_family"
    ]
    == "tool_use_error"
).astype(int)

print(
    "Support:",
    len(subset_tc_tc)
)

print(
    "True tool-use:",
    subset_tc_tc[
        "target_tool_use"
    ].sum()
)

print(
    "Prevalence:",
    subset_tc_tc[
        "target_tool_use"
    ].mean()
)

Support: 407
True tool-use: 72
Prevalence: 0.1769041769041769


In [52]:
# ============================================================
# 33B. TOOL_CALL -> ASSISTANT
#      Semantic workflow prediction:
#      is it actually grounding_state_error?
# ============================================================

subset_tc_asst = workflow_cases[
    workflow_cases[
        "transition_type"
    ]
    == "TOOL_CALL->ASSISTANT"
].copy()

subset_tc_asst[
    "target_grounding"
] = (
    subset_tc_asst[
        "true_family"
    ]
    == "grounding_state_error"
).astype(int)

print(
    "Support:",
    len(subset_tc_asst)
)

print(
    "True grounding:",
    subset_tc_asst[
        "target_grounding"
    ].sum()
)

print(
    "Prevalence:",
    subset_tc_asst[
        "target_grounding"
    ].mean()
)

Support: 190
True grounding: 39
Prevalence: 0.20526315789473684


In [53]:
# ============================================================
# 33C. ASSISTANT -> TOOL_CALL
#      Semantic workflow prediction:
#      is it actually constraint_error?
# ============================================================

subset_asst_tc = workflow_cases[
    workflow_cases[
        "transition_type"
    ]
    == "ASSISTANT->TOOL_CALL"
].copy()

subset_asst_tc[
    "target_constraint"
] = (
        subset_asst_tc[
            "true_family"
        ]
        == "constraint_error"
    ).astype(int)

print(
    "Support:",
    len(subset_asst_tc)
)

print(
    "True constraint:",
    subset_asst_tc[
        "target_constraint"
    ].sum()
)

print(
    "Prevalence:",
    subset_asst_tc[
        "target_constraint"
    ].mean()
)

Support: 139
True constraint: 30
Prevalence: 0.2158273381294964


This is a strong result because it reveals the shape of the refinement problem much more clearly.

The no-t1 model predicts workflow_error 873 times, but only 483/873 = 55.3% of those are actually workflow errors. The remaining 390 cases are hidden local failure types:

146 constraint errors
131 tool-use errors
107 grounding-state errors
6 reasoning-value errors

So workflow_error is acting like a catch-all basin for ambiguous cases.

At the same time, t1 almost never creates workflow predictions:

changed predictions: 181
moves into workflow: 0
moves out of workflow: 159

That is extremely important. The trajectory signal is behaving almost purely as a refinement operator:

semantic/no-history model says “workflow”

trajectory evidence says “this looks more specifically like tool-use / grounding / constraint”

And the transition type tells us which refinement is most plausible.

The three mechanism-specific subproblems are also large enough to model:

Mechanism	Support	Positive target	Prevalence
TOOL_CALL→TOOL_CALL: workflow → tool-use	407	72	17.7%
TOOL_CALL→ASSISTANT: workflow → grounding	190	39	20.5%
ASSISTANT→TOOL_CALL: workflow → constraint	139	30	21.6%

These are not huge datasets, but they are large enough for small calibrated binary models with strict cross-fitting.

The current trajectory model captures only part of the available opportunity:

TOOL_CALL→TOOL_CALL: 31/125 = 24.8%
ASSISTANT→TOOL_CALL: 19/70 = 27.1%
TOOL_CALL→ASSISTANT: 22/117 = 18.8%
ASSISTANT→ASSISTANT: only 6.3%

That suggests the next research step should not be another 5-class classifier. It should be specialized refinement detectors.

Next research hypothesis

Instead of:

P(failure family∣x)

build:

P(refine workflow→k∣transition type,semantic probabilities,trajectory evidence)

for a small number of mechanism-specific targets.

The first three experts should be:

TOOL_CALL → TOOL_CALL
workflow_error → tool_use_error


TOOL_CALL → ASSISTANT
workflow_error → grounding_state_error


ASSISTANT → TOOL_CALL
workflow_error → constraint_error

This is now a genuinely different architecture from the old MoE.

The old router asked:

which complete classifier should win?

This asks:

is this particular workflow prediction actually a specific local failure?

That is a much easier and more interpretable decision

In [54]:
# ============================================================
# 34. Build mechanism-specific refinement datasets
# ============================================================

# Probability outputs BEFORE t-1 is injected
base_prob = remove_t1_prob

# Probability outputs WITH t-1
traj_prob = full_prob

class_to_idx = {
    name: i
    for i, name in enumerate(CLASS_NAMES)
}


def build_refinement_dataset(
    transition,
    target_family,
):
    """
    Restrict to cases where:
      1. no-t1 model predicts workflow_error
      2. transition matches requested mechanism

    Binary target:
      1 = true label is target_family
      0 = otherwise
    """

    mask = (
        (mechanism_df["transition_type"] == transition)
        &
        (
            mechanism_df["no_t1_prediction"]
            == "workflow_error"
        )
    )

    idx = np.where(mask)[0]

    target_id = class_to_idx[
        target_family
    ]

    y_binary = (
        y_train[idx]
        == target_id
    ).astype(int)

    return idx, y_binary


refinement_tasks = {
    "tool_to_tool__tool_use": (
        "TOOL_CALL->TOOL_CALL",
        "tool_use_error",
    ),

    "tool_to_assistant__grounding": (
        "TOOL_CALL->ASSISTANT",
        "grounding_state_error",
    ),

    "assistant_to_tool__constraint": (
        "ASSISTANT->TOOL_CALL",
        "constraint_error",
    ),
}


refinement_indices = {}

for task, (
    transition,
    target_family,
) in refinement_tasks.items():

    idx, y_bin = build_refinement_dataset(
        transition,
        target_family,
    )

    refinement_indices[task] = (
        idx,
        y_bin,
    )

    print(
        task,
        "| n =", len(idx),
        "| positives =", y_bin.sum(),
        "| prevalence =", y_bin.mean(),
    )

tool_to_tool__tool_use | n = 407 | positives = 72 | prevalence = 0.1769041769041769
tool_to_assistant__grounding | n = 190 | positives = 39 | prevalence = 0.20526315789473684
assistant_to_tool__constraint | n = 139 | positives = 30 | prevalence = 0.2158273381294964


In [55]:
# ============================================================
# 35. Mechanism-specific refinement features
# ============================================================

def build_refinement_features(
    idx,
    target_family,
):

    workflow_id = class_to_idx[
        "workflow_error"
    ]

    target_id = class_to_idx[
        target_family
    ]

    base_workflow = (
        base_prob[idx, workflow_id]
    )

    base_target = (
        base_prob[idx, target_id]
    )

    traj_workflow = (
        traj_prob[idx, workflow_id]
    )

    traj_target = (
        traj_prob[idx, target_id]
    )

    X = pd.DataFrame({
        "base_workflow_prob":
            base_workflow,

        "base_target_prob":
            base_target,

        "base_target_minus_workflow":
            base_target
            - base_workflow,

        "base_confidence":
            uncertainty_df.loc[
                idx,
                "confidence"
            ].to_numpy(),

        "base_margin":
            uncertainty_df.loc[
                idx,
                "margin"
            ].to_numpy(),

        "base_entropy":
            uncertainty_df.loc[
                idx,
                "entropy"
            ].to_numpy(),

        "t1_l1":
            distance_df.loc[
                idx,
                "current_tminus1_l1"
            ].to_numpy(),

        "t1_l2":
            distance_df.loc[
                idx,
                "current_tminus1_l2"
            ].to_numpy(),

        "target_prob_gain":
            traj_target
            - base_target,

        "workflow_prob_drop":
            base_workflow
            - traj_workflow,

        "relative_target_gain":
            (
                traj_target
                - base_target
            )
            /
            (
                base_target
                + 1e-8
            ),

        "traj_target_minus_workflow":
            traj_target
            - traj_workflow,
    })

    return X

In [56]:
for task, (
    transition,
    target_family,
) in refinement_tasks.items():

    idx, y_bin = (
        refinement_indices[task]
    )

    X_task = build_refinement_features(
        idx,
        target_family,
    )

    print(
        "\n",
        "=" * 70,
        "\n",
        task,
        X_task.shape,
    )

    display(
        X_task.groupby(
            y_bin
        ).mean().round(4)
    )


 tool_to_tool__tool_use (407, 12)


,base_workflow_prob,base_target_prob,base_target_minus_workflow,base_confidence,base_margin,base_entropy,t1_l1,t1_l2,target_prob_gain,workflow_prob_drop,relative_target_gain,traj_target_minus_workflow
0,0.8158,0.0841,-0.7317,0.8158,0.7006,0.5532,0.0223,0.5482,0.0183,0.040,0.2322,-0.6735
1,0.6254,0.2669,-0.3586,0.6254,0.3397,0.8530,0.0315,0.7803,0.0601,0.078,0.2530,-0.2204



 tool_to_assistant__grounding (190, 12)


,base_workflow_prob,base_target_prob,base_target_minus_workflow,base_confidence,base_margin,base_entropy,t1_l1,t1_l2,target_prob_gain,workflow_prob_drop,relative_target_gain,traj_target_minus_workflow
0,0.6197,0.0777,-0.542,0.6197,0.3889,0.9655,0.0438,1.0806,0.0485,0.0945,0.7278,-0.3990
1,0.5401,0.1721,-0.368,0.5401,0.2873,1.0924,0.0457,1.1240,0.0941,0.1153,0.6208,-0.1586



 assistant_to_tool__constraint (139, 12)


,base_workflow_prob,base_target_prob,base_target_minus_workflow,base_confidence,base_margin,base_entropy,t1_l1,t1_l2,target_prob_gain,workflow_prob_drop,relative_target_gain,traj_target_minus_workflow
0,0.6839,0.1081,-0.5758,0.6839,0.4889,0.8000,0.0488,1.2035,0.0037,0.1003,0.0904,-0.4718
1,0.6185,0.2215,-0.3970,0.6185,0.3691,0.9283,0.0449,1.1070,0.0245,0.0986,0.1131,-0.2738


In [57]:
# ============================================================
# 36. Cross-fitted specialized refinement experts
# ============================================================

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_fscore_support,
)

refinement_results = {}
refinement_oof_prob = {}


for task, (
    transition,
    target_family,
) in refinement_tasks.items():

    idx, y_bin = (
        refinement_indices[task]
    )

    X_task = (
        build_refinement_features(
            idx,
            target_family,
        )
        .to_numpy(
            dtype=np.float32
        )
    )

    task_groups = (
        groups_train[idx]
    )

    # number of folds may need reduction
    # if positives per fold become too small
    task_cv = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=42,
    )

    oof_prob = np.zeros(
        len(idx),
        dtype=float,
    )

    for fold, (
        tr_local,
        va_local,
    ) in enumerate(
        task_cv.split(
            X_task,
            y_bin,
            groups=task_groups,
        ),
        start=1,
    ):

        scaler = StandardScaler()

        X_tr = scaler.fit_transform(
            X_task[tr_local]
        )

        X_va = scaler.transform(
            X_task[va_local]
        )

        model = LogisticRegression(
            C=0.1,
            class_weight="balanced",
            max_iter=5000,
            random_state=42,
        )

        model.fit(
            X_tr,
            y_bin[tr_local],
        )

        oof_prob[
            va_local
        ] = model.predict_proba(
            X_va
        )[:, 1]

    refinement_oof_prob[
        task
    ] = oof_prob

    pr_auc = average_precision_score(
        y_bin,
        oof_prob,
    )

    roc_auc = roc_auc_score(
        y_bin,
        oof_prob,
    )

    refinement_results[
        task
    ] = {
        "support":
            len(y_bin),

        "positives":
            int(y_bin.sum()),

        "prevalence":
            float(y_bin.mean()),

        "pr_auc":
            pr_auc,

        "roc_auc":
            roc_auc,
    }


display(
    pd.DataFrame(
        refinement_results
    ).T.round(4)
)

,support,positives,prevalence,pr_auc,roc_auc
tool_to_tool__tool_use,407.0,72.0,0.1769,0.4541,0.8224
tool_to_assistant__grounding,190.0,39.0,0.2053,0.4898,0.6962
assistant_to_tool__constraint,139.0,30.0,0.2158,0.5135,0.7177


In [58]:
ROUTER_FEATURES = [
    "base_workflow_prob",
    "base_target_prob",
    "base_target_minus_workflow",
    "base_confidence",
    "base_margin",
    "base_entropy",

    "t1_l1",
    "t1_l2",

    "target_prob_gain",
    "workflow_prob_drop",
    "relative_target_gain",
    "traj_target_minus_workflow",
]

EVIDENCE_FEATURES = [
    "base_workflow_prob",
    "base_target_prob",
    "base_target_minus_workflow",
    "base_confidence",
    "base_margin",
    "base_entropy",

    "t1_l1",
    "t1_l2",
]

1. Main result

The useful comparison is not PR-AUC against 0.5; it is PR-AUC against each task's positive prevalence.

Refinement task	Prevalence	PR-AUC	PR-AUC / baseline	ROC-AUC
Tool→Tool → tool-use	0.1769	0.4541	2.57×	0.8224
Tool→Assistant → grounding	0.2053	0.4898	2.39×	0.6962
Assistant→Tool → constraint	0.2158	0.5135	2.38×	0.7177

All three specialized refinement problems contain substantial predictive signal.

Especially important is:

Tool→Tool / tool-use is highly separable.

ROC-AUC 0.8224 on 407 examples is considerably stronger than anything we were getting by asking the trajectory features to solve the whole 5-class problem.

That supports the architecture we've been converging toward:

semantic family classifier→detect ambiguous workflow region→transition-conditioned refinement expert

rather than:

[semantic+trajectory]→one flat 5-class classifier.
2. The feature means tell an even more interesting story
Tool→Tool: workflow vs tool-use

For true tool-use errors:

base workflow probability:  0.625 vs 0.816
base tool-use probability:  0.267 vs 0.084
entropy:                    0.853 vs 0.553


t1 L1:                      0.0315 vs 0.0223
t1 L2:                      0.780  vs 0.548


target probability gain:    0.060 vs 0.018
workflow probability drop:  0.078 vs 0.040

This is excellent mechanistically.

Even before trajectory is added, the semantic model already partially suspects tool-use. It just doesn't have enough evidence to defeat workflow_error.

Then t1 supplies additional relational evidence.

So trajectory isn't discovering tool-use from nowhere.

It's doing something closer to:

semantic model: "probably workflow, but tool-use is plausible"

trajectory: "the recent tool→tool relation provides evidence that the local tool-use explanation is better."

That is exactly the kind of expert-routing behavior we wanted.

3. Tool→Assistant grounding is different

The separation is weaker:

ROC-AUC = 0.696
but PR-AUC = 0.490 against prevalence 0.205

There is still useful signal.

True grounding errors have:

workflow probability
0.540 vs 0.620


grounding probability
0.172 vs 0.078


entropy
1.092 vs 0.966


target probability gain
0.094 vs 0.049

Again, semantic ambiguity matters.

But notice something important:

t1 L1: 0.0457 vs 0.0438
t1 L2: 1.1240 vs 1.0806

Raw displacement barely separates the classes.

That's consistent with your earlier AUC result:

grounding L1 AUC ≈ 0.522
grounding L2 AUC ≈ 0.519

Therefore:

Grounding is probably not encoded in distance magnitude.

The useful information likely lies in the semantic relation between the tool result and assistant response.

This gives us a very concrete later feature-engineering target.

4. Assistant→Tool constraint is also learnable

PR-AUC:

0.5135

against prevalence:

0.2158

That's strong.

And again the semantic classifier already contains the beginning of the answer:

non-constraint:
workflow = 0.684
constraint = 0.108


constraint:
workflow = 0.619
constraint = 0.222

So this expert is probably detecting:

"semantic classifier is uncertain between workflow and constraint, and the assistant→tool relation tells us which interpretation is correct."

This is much more promising than simply feeding t1_l1 and t1_l2 into the global classifier.

In [59]:
# ============================================================
# 37. Refinement expert feature-family ablation
# ============================================================

REFINEMENT_FEATURE_SETS = {

    # What did the semantic classifier believe?
    "semantic_state": [
        "base_workflow_prob",
        "base_target_prob",
        "base_target_minus_workflow",
        "base_confidence",
        "base_margin",
        "base_entropy",
    ],

    # Does raw displacement itself identify the refinement?
    "distance_only": [
        "t1_l1",
        "t1_l2",
    ],

    # Can semantic ambiguity + raw relation solve it?
    "semantic_plus_distance": [
        "base_workflow_prob",
        "base_target_prob",
        "base_target_minus_workflow",
        "base_confidence",
        "base_margin",
        "base_entropy",
        "t1_l1",
        "t1_l2",
    ],

    # Can we route based on how trajectory changes the classifier?
    "full_router": [
        "base_workflow_prob",
        "base_target_prob",
        "base_target_minus_workflow",
        "base_confidence",
        "base_margin",
        "base_entropy",
        "t1_l1",
        "t1_l2",
        "target_prob_gain",
        "workflow_prob_drop",
        "relative_target_gain",
        "traj_target_minus_workflow",
    ],
}

In [60]:
# ============================================================
# 38. Cross-fitted refinement ablation
# ============================================================

ablation_rows = []

for task, (transition, target_family) in refinement_tasks.items():

    idx, y_bin = refinement_indices[task]

    X_all = build_refinement_features(
        idx,
        target_family,
    )

    task_groups = groups_train[idx]

    for feature_set_name, columns in REFINEMENT_FEATURE_SETS.items():

        X_task = X_all[
            columns
        ].to_numpy(dtype=np.float32)

        oof_prob = np.zeros(
            len(idx),
            dtype=float,
        )

        task_cv = StratifiedGroupKFold(
            n_splits=5,
            shuffle=True,
            random_state=42,
        )

        for tr_local, va_local in task_cv.split(
            X_task,
            y_bin,
            groups=task_groups,
        ):

            scaler = StandardScaler()

            X_tr = scaler.fit_transform(
                X_task[tr_local]
            )

            X_va = scaler.transform(
                X_task[va_local]
            )

            model = LogisticRegression(
                C=0.1,
                class_weight="balanced",
                max_iter=5000,
                random_state=42,
            )

            model.fit(
                X_tr,
                y_bin[tr_local],
            )

            oof_prob[va_local] = (
                model.predict_proba(X_va)[:, 1]
            )

        ablation_rows.append({
            "task": task,
            "feature_set": feature_set_name,
            "n_features": len(columns),
            "support": len(y_bin),
            "positives": int(y_bin.sum()),
            "prevalence": y_bin.mean(),

            "pr_auc":
                average_precision_score(
                    y_bin,
                    oof_prob,
                ),

            "roc_auc":
                roc_auc_score(
                    y_bin,
                    oof_prob,
                ),
        })


refinement_ablation_df = pd.DataFrame(
    ablation_rows
)

display(
    refinement_ablation_df
    .sort_values(
        ["task", "pr_auc"],
        ascending=[True, False],
    )
    .round(4)
)

,task,feature_set,n_features,support,positives,prevalence,pr_auc,roc_auc
11,assistant_to_tool__constraint,full_router,12,139,30,0.2158,0.5135,0.7177
8,assistant_to_tool__constraint,semantic_state,6,139,30,0.2158,0.4471,0.7015
10,assistant_to_tool__constraint,semantic_plus_distance,8,139,30,0.2158,0.4100,0.7107
9,assistant_to_tool__constraint,distance_only,2,139,30,0.2158,0.3528,0.6480
6,tool_to_assistant__grounding,semantic_plus_distance,8,190,39,0.2053,0.5075,0.6972
4,tool_to_assistant__grounding,semantic_state,6,190,39,0.2053,0.5053,0.7156
7,tool_to_assistant__grounding,full_router,12,190,39,0.2053,0.4898,0.6962
5,tool_to_assistant__grounding,distance_only,2,190,39,0.2053,0.2541,0.5444
2,tool_to_tool__tool_use,semantic_plus_distance,8,407,72,0.1769,0.4734,0.8243
0,tool_to_tool__tool_use,semantic_state,6,407,72,0.1769,0.4593,0.8362


In [61]:
display(
    refinement_ablation_df.pivot(
        index="task",
        columns="feature_set",
        values="pr_auc",
    ).round(4)
)

display(
    refinement_ablation_df.pivot(
        index="task",
        columns="feature_set",
        values="roc_auc",
    ).round(4)
)

feature_set,distance_only,full_router,semantic_plus_distance,semantic_state
task,,,,
assistant_to_tool__constraint,0.3528,0.5135,0.4100,0.4471
tool_to_assistant__grounding,0.2541,0.4898,0.5075,0.5053
tool_to_tool__tool_use,0.2590,0.4541,0.4734,0.4593


feature_set,distance_only,full_router,semantic_plus_distance,semantic_state
task,,,,
assistant_to_tool__constraint,0.6480,0.7177,0.7107,0.7015
tool_to_assistant__grounding,0.5444,0.6962,0.6972,0.7156
tool_to_tool__tool_use,0.6417,0.8224,0.8243,0.8362


7. What this experiment will tell us

There are several possible outcomes.

A. semantic_state is already almost as good as full_router

Suppose:

semantic       PR-AUC .44
distance       PR-AUC .20
semantic+dist  PR-AUC .45
full router    PR-AUC .46

Then trajectory isn't really providing the mechanism.

The semantic classifier already knows that these are ambiguous workflow cases.

That would mean we should build hierarchical semantic classification, not trajectory experts.

B. distance_only is weak, but semantic+distance improves

That would be particularly interesting.

It would mean:

trajectory utility=f(semantic state,trajectory)

rather than trajectory having standalone predictive value.

That's a true interaction.

This would strongly support MoE/routing.

C. Only full_router performs strongly

For example:

semantic         .30
distance         .20
semantic+dist    .32
full_router      .51

Then the correct architecture is:

Semantic expert
       ↓
Trajectory expert
       ↓
Correction router
       ↓
Final prediction

The router isn't learning the failure directly.

It is learning:

when to trust the trajectory expert's disagreement with the semantic expert.

That's still a very valuable reliability architecture.

D. distance_only is strong for Tool→Tool

This would be surprising given our previous experiments, but very useful.

It would mean tool-use failure has an actual geometric signature in recent trajectory displacement.

Then we investigate that geometry directly.

8. I expect different mechanisms to produce different answers

Based on everything you've found so far, my current expectation is:

Tool→Tool / tool-use

semantic_state          strong
distance_only           moderate
semantic+distance       stronger
full_router             strongest

This looks like the best candidate for a real semantic × trajectory interaction.

Tool→Assistant / grounding

semantic_state          moderate
distance_only           weak
semantic+distance       small improvement
full_router             stronger

I don't think L1/L2 is the correct representation for grounding. Eventually we probably need explicit tool-result ↔ assistant-response consistency features.

Assistant→Tool / constraint

semantic_state          fairly strong
distance_only           weak/moderate
semantic+distance       moderate
full_router             strong

Eventually this should probably become an assistant-intent ↔ tool-call compliance expert.

And this changes how we should think about "hallucination"

Your earlier question about hallucinations after correct tool calls fits directly here.

A TOOL_CALL→ASSISTANT transition can have multiple failure mechanisms:

correct tool result
       ↓
assistant response
       ↓
grounding failure / hallucination

Distance from the assistant representation to the previous event may tell us something changed, but not what relationship was violated.

For grounding, what we actually want eventually is something like:

R(E
tool result
	​

,E
assistant response
	​

)

not merely:

∥E
assistant
	​

−E
tool
	​

∥

Similarly:

R(E
assistant intent
	​

,E
tool call
	​

)

for constraint/tool-selection failures.

That's why Notebook 16 is now pointing toward relation-aware reliability experts, rather than generic temporal distance.

But don't jump there yet. Run Cells 37–38 first.

If full_router >> semantic_plus_distance, we've learned that model disagreement itself is the useful signal. If semantic_plus_distance is competitive with full_router, then we have evidence for a genuinely learnable trajectory mechanism and should move next into relation-specific representations.

This ablation is very informative because it shows the three refinement tasks are not driven by the same mechanism.

For tool→tool / tool_use, the strongest signal is already in the semantic state: ROC-AUC 0.8362 and PR-AUC 0.4593. Adding distance gives only a small PR-AUC gain to 0.4734, while the full router actually drops slightly to 0.4541. So for this branch, trajectory distance is secondary. The semantic model already “knows” many of these are tool-use-like cases but still outputs workflow. That points toward hierarchical semantic refinement, not a trajectory-heavy expert.

For tool→assistant / grounding, the same pattern is even clearer. semantic_state alone gives PR-AUC 0.5053 and the best ROC-AUC, 0.7156. Adding distance barely changes PR-AUC to 0.5075, and the full router is worse at 0.4898. Raw distance alone is weak at 0.2541 PR-AUC. So the grounding branch is also mostly semantic under the current feature representation. This reinforces the earlier conclusion that generic t−1 distance is not the right grounding signal.

The interesting exception is assistant→tool / constraint. Here the full router is clearly best: PR-AUC 0.5135 versus 0.4471 for semantic state, while semantic+distance actually falls to 0.4100. That means the useful extra information is not simply raw t−1 distance. It is more likely the way the trajectory/full model changes its probability distribution. In other words, this branch really does look like a meta-routing problem.

So the current architectural picture is becoming:

workflow-like semantic basin
        |
        +-- TOOL -> TOOL
        |      mostly semantic refinement -> tool_use
        |
        +-- TOOL -> ASSISTANT
        |      mostly semantic refinement -> grounding
        |      later add better evidence-grounding features
        |
        +-- ASSISTANT -> TOOL
               semantic + model-disagreement/router signal -> constraint

I would not build one shared refinement router across all three tasks. The ablation says that would mix different mechanisms.

The next experiment should therefore test how much each specialized binary expert can improve the original five-class predictions at realistic operating thresholds, with threshold selection cross-fitted and precision/rescue-break tradeoffs reported. That tells us whether these good PR-AUCs translate into actual end-to-end reliability gains.

In [63]:
# ============================================================
# 39. Best feature set per specialized refinement expert
# ============================================================

BEST_REFINEMENT_FEATURES = {
    "tool_to_tool__tool_use":
        REFINEMENT_FEATURE_SETS["semantic_plus_distance"],

    "tool_to_assistant__grounding":
        REFINEMENT_FEATURE_SETS["semantic_state"],

    "assistant_to_tool__constraint":
        REFINEMENT_FEATURE_SETS["full_router"],
}

REFINEMENT_TARGETS = {
    "tool_to_tool__tool_use":
        "tool_use_error",

    "tool_to_assistant__grounding":
        "grounding_state_error",

    "assistant_to_tool__constraint":
        "constraint_error",
}

for task in BEST_REFINEMENT_FEATURES:
    print(
        task,
        "->",
        REFINEMENT_TARGETS[task],
        "| features =",
        len(BEST_REFINEMENT_FEATURES[task]),
    )

tool_to_tool__tool_use -> tool_use_error | features = 8
tool_to_assistant__grounding -> grounding_state_error | features = 6
assistant_to_tool__constraint -> constraint_error | features = 12


In [65]:
# ============================================================
# 40. Cross-fitted specialist probabilities
# ============================================================

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

specialist_oof_prob = {}
specialist_summary = []

for task, (
    transition,
    target_family,
) in refinement_tasks.items():

    idx, y_bin = refinement_indices[task]

    X_all = build_refinement_features(
        idx,
        target_family,
    )

    feature_cols = (
        BEST_REFINEMENT_FEATURES[task]
    )

    X_task = X_all[
        feature_cols
    ].to_numpy(dtype=np.float32)

    task_groups = groups_train[idx]

    oof_prob = np.zeros(
        len(idx),
        dtype=float,
    )

    task_cv = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=42,
    )

    for fold, (
        tr_local,
        va_local,
    ) in enumerate(
        task_cv.split(
            X_task,
            y_bin,
            groups=task_groups,
        ),
        start=1,
    ):

        scaler = StandardScaler()

        X_tr = scaler.fit_transform(
            X_task[tr_local]
        )

        X_va = scaler.transform(
            X_task[va_local]
        )

        model = LogisticRegression(
            C=0.1,
            class_weight="balanced",
            max_iter=5000,
            random_state=42,
        )

        model.fit(
            X_tr,
            y_bin[tr_local],
        )

        oof_prob[
            va_local
        ] = model.predict_proba(
            X_va
        )[:, 1]

    specialist_oof_prob[
        task
    ] = oof_prob

    specialist_summary.append({
        "task":
            task,

        "support":
            len(y_bin),

        "positives":
            int(y_bin.sum()),

        "prevalence":
            y_bin.mean(),

        "pr_auc":
            average_precision_score(
                y_bin,
                oof_prob,
            ),

        "roc_auc":
            roc_auc_score(
                y_bin,
                oof_prob,
            ),
    })

specialist_summary_df = pd.DataFrame(
    specialist_summary
)

display(
    specialist_summary_df.round(4)
)

,task,support,positives,prevalence,pr_auc,roc_auc
0,tool_to_tool__tool_use,407,72,0.1769,0.4734,0.8243
1,tool_to_assistant__grounding,190,39,0.2053,0.5053,0.7156
2,assistant_to_tool__constraint,139,30,0.2158,0.5135,0.7177


In [66]:
# ============================================================
# 41. Specialist override threshold sweeps
# ============================================================

threshold_grid = np.arange(
    0.30,
    0.91,
    0.025,
)

specialist_threshold_rows = []

base_global_pred = remove_t1_pred.copy()
base_global_correct = (
    base_global_pred == y_train
)

for task, (
    transition,
    target_family,
) in refinement_tasks.items():

    idx, y_bin = refinement_indices[task]

    probs = specialist_oof_prob[
        task
    ]

    target_id = class_to_idx[
        target_family
    ]

    for threshold in threshold_grid:

        selected_local = (
            probs >= threshold
        )

        selected_idx = idx[
            selected_local
        ]

        pred = (
            base_global_pred.copy()
        )

        pred[
            selected_idx
        ] = target_id

        correct = (
            pred == y_train
        )

        rescues = (
            (~base_global_correct)
            &
            correct
        )

        breaks = (
            base_global_correct
            &
            (~correct)
        )

        overrides = int(
            selected_local.sum()
        )

        good = int(
            (
                y_train[selected_idx]
                == target_id
            ).sum()
        )

        bad = (
            overrides - good
        )

        specialist_threshold_rows.append({
            "task":
                task,

            "target_family":
                target_family,

            "threshold":
                threshold,

            "overrides":
                overrides,

            "good":
                good,

            "bad":
                bad,

            "precision":
                (
                    good / overrides
                    if overrides > 0
                    else np.nan
                ),

            "rescues":
                int(rescues.sum()),

            "breaks":
                int(breaks.sum()),

            "net":
                int(
                    rescues.sum()
                    -
                    breaks.sum()
                ),

            **metrics_local(pred),
        })

specialist_threshold_df = pd.DataFrame(
    specialist_threshold_rows
)

In [67]:
# ============================================================
# 42. Best diagnostic thresholds per specialist
# ============================================================

for task in refinement_tasks:

    print(
        "\n",
        "=" * 80,
        "\n",
        task,
    )

    display(
        specialist_threshold_df[
            specialist_threshold_df[
                "task"
            ] == task
        ]
        .sort_values(
            [
                "macro_f1",
                "accuracy",
                "net",
            ],
            ascending=False,
        )
        .head(12)
        .round(4)
    )


 tool_to_tool__tool_use


,task,target_family,threshold,overrides,good,bad,precision,rescues,breaks,net,accuracy,balanced_accuracy,macro_f1,weighted_f1
11,tool_to_tool__tool_use,tool_use_error,0.575,109,49,60,0.4495,49,41,8,0.5205,0.4644,0.4769,0.5118
7,tool_to_tool__tool_use,tool_use_error,0.475,134,55,79,0.4104,55,49,6,0.5191,0.4671,0.4767,0.5116
10,tool_to_tool__tool_use,tool_use_error,0.550,112,49,63,0.4375,49,44,5,0.5185,0.4635,0.4757,0.5101
6,tool_to_tool__tool_use,tool_use_error,0.450,139,55,84,0.3957,55,51,4,0.5178,0.4665,0.4756,0.5105
9,tool_to_tool__tool_use,tool_use_error,0.525,117,50,67,0.4274,50,46,4,0.5178,0.4638,0.4754,0.5096
8,tool_to_tool__tool_use,tool_use_error,0.500,127,52,75,0.4094,52,48,4,0.5178,0.4648,0.4753,0.5100
16,tool_to_tool__tool_use,tool_use_error,0.700,78,39,39,0.5000,39,31,8,0.5205,0.4590,0.4749,0.5104
12,tool_to_tool__tool_use,tool_use_error,0.600,101,45,56,0.4455,45,41,4,0.5178,0.4611,0.4745,0.5089
17,tool_to_tool__tool_use,tool_use_error,0.725,69,36,33,0.5217,36,28,8,0.5205,0.4574,0.4742,0.5099
5,tool_to_tool__tool_use,tool_use_error,0.425,147,56,91,0.3810,56,56,0,0.5151,0.4658,0.4741,0.5083



 tool_to_assistant__grounding


,task,target_family,threshold,overrides,good,bad,precision,rescues,breaks,net,accuracy,balanced_accuracy,macro_f1,weighted_f1
32,tool_to_assistant__grounding,grounding_state_error,0.475,69,27,42,0.3913,27,19,8,0.5205,0.4519,0.4679,0.5087
34,tool_to_assistant__grounding,grounding_state_error,0.525,56,23,33,0.4107,23,13,10,0.5218,0.4504,0.4673,0.5089
36,tool_to_assistant__grounding,grounding_state_error,0.575,44,20,24,0.4545,20,8,12,0.5232,0.4495,0.4672,0.5093
35,tool_to_assistant__grounding,grounding_state_error,0.550,51,21,30,0.4118,21,10,11,0.5225,0.4497,0.4668,0.5091
33,tool_to_assistant__grounding,grounding_state_error,0.500,59,23,36,0.3898,23,15,8,0.5205,0.4498,0.4665,0.5078
37,tool_to_assistant__grounding,grounding_state_error,0.600,37,18,19,0.4865,18,7,11,0.5225,0.4481,0.4664,0.5081
31,tool_to_assistant__grounding,grounding_state_error,0.450,77,27,50,0.3506,27,22,5,0.5185,0.4510,0.4663,0.5072
30,tool_to_assistant__grounding,grounding_state_error,0.425,82,27,55,0.3293,27,24,3,0.5171,0.4504,0.4653,0.5062
38,tool_to_assistant__grounding,grounding_state_error,0.625,33,16,17,0.4848,16,7,9,0.5212,0.4465,0.4650,0.5065
29,tool_to_assistant__grounding,grounding_state_error,0.400,95,29,66,0.3053,29,29,0,0.5151,0.4505,0.4645,0.5052



 assistant_to_tool__constraint


,task,target_family,threshold,overrides,good,bad,precision,rescues,breaks,net,accuracy,balanced_accuracy,macro_f1,weighted_f1
63,assistant_to_tool__constraint,constraint_error,0.625,27,14,13,0.5185,14,8,6,0.5191,0.4419,0.4602,0.5028
64,assistant_to_tool__constraint,constraint_error,0.650,27,14,13,0.5185,14,8,6,0.5191,0.4419,0.4602,0.5028
60,assistant_to_tool__constraint,constraint_error,0.550,40,17,23,0.4250,17,13,4,0.5178,0.4423,0.4600,0.5023
61,assistant_to_tool__constraint,constraint_error,0.575,35,16,19,0.4571,16,12,4,0.5178,0.4420,0.4599,0.5021
62,assistant_to_tool__constraint,constraint_error,0.600,28,14,14,0.5000,14,9,5,0.5185,0.4416,0.4599,0.5022
58,assistant_to_tool__constraint,constraint_error,0.500,48,19,29,0.3958,19,18,1,0.5158,0.4420,0.4595,0.5010
65,assistant_to_tool__constraint,constraint_error,0.675,23,12,11,0.5217,12,7,5,0.5185,0.4410,0.4594,0.5019
59,assistant_to_tool__constraint,constraint_error,0.525,42,17,25,0.4048,17,15,2,0.5165,0.4417,0.4594,0.5012
66,assistant_to_tool__constraint,constraint_error,0.700,18,11,7,0.6111,11,6,5,0.5185,0.4406,0.4593,0.5016
53,assistant_to_tool__constraint,constraint_error,0.375,70,24,46,0.3429,24,27,-3,0.5131,0.4425,0.4590,0.4998


In [68]:
# ============================================================
# 43. Cross-fitted threshold selection for each specialist
# ============================================================

specialist_crossfit_choice = {}
specialist_threshold_selection = []

for task, (
    transition,
    target_family,
) in refinement_tasks.items():

    idx, y_bin = refinement_indices[task]

    probs = specialist_oof_prob[
        task
    ]

    task_groups = groups_train[
        idx
    ]

    target_id = class_to_idx[
        target_family
    ]

    chosen_local = np.zeros(
        len(idx),
        dtype=bool,
    )

    threshold_cv = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=123,
    )

    for fold, (
        tr_local,
        va_local,
    ) in enumerate(
        threshold_cv.split(
            np.zeros(
                (len(idx), 1)
            ),
            y_bin,
            groups=task_groups,
        ),
        start=1,
    ):

        candidates = []

        for threshold in threshold_grid:

            train_selected = (
                probs[
                    tr_local
                ]
                >= threshold
            )

            train_idx_global = idx[
                tr_local
            ]

            pred_train = (
                base_global_pred[
                    train_idx_global
                ].copy()
            )

            pred_train[
                train_selected
            ] = target_id

            y_train_local = y_train[
                train_idx_global
            ]

            candidates.append({
                "threshold":
                    threshold,

                "macro_f1":
                    f1_score(
                        y_train_local,
                        pred_train,
                        average="macro",
                        zero_division=0,
                    ),

                "accuracy":
                    accuracy_score(
                        y_train_local,
                        pred_train,
                    ),

                "selected":
                    int(
                        train_selected.sum()
                    ),
            })

        candidate_df = pd.DataFrame(
            candidates
        )

        best = (
            candidate_df
            .sort_values(
                [
                    "macro_f1",
                    "accuracy",
                ],
                ascending=False,
            )
            .iloc[0]
        )

        threshold = float(
            best[
                "threshold"
            ]
        )

        eval_selected = (
            probs[
                va_local
            ]
            >= threshold
        )

        chosen_local[
            va_local
        ] = eval_selected

        specialist_threshold_selection.append({
            "task":
                task,

            "fold":
                fold,

            "selected_threshold":
                threshold,

            "train_macro_f1":
                best[
                    "macro_f1"
                ],

            "eval_overrides":
                int(
                    eval_selected.sum()
                ),
        })

    specialist_crossfit_choice[
        task
    ] = chosen_local

In [69]:
# ============================================================
# 44. Threshold stability
# ============================================================

specialist_threshold_selection_df = (
    pd.DataFrame(
        specialist_threshold_selection
    )
)

display(
    specialist_threshold_selection_df
)

display(
    specialist_threshold_selection_df
    .groupby("task")
    .agg(
        mean_threshold=(
            "selected_threshold",
            "mean",
        ),

        median_threshold=(
            "selected_threshold",
            "median",
        ),

        min_threshold=(
            "selected_threshold",
            "min",
        ),

        max_threshold=(
            "selected_threshold",
            "max",
        ),

        mean_eval_overrides=(
            "eval_overrides",
            "mean",
        ),
    )
    .round(4)
)

,task,fold,selected_threshold,train_macro_f1,eval_overrides
0,tool_to_tool__tool_use,1,0.700,0.318596,7
1,tool_to_tool__tool_use,2,0.475,0.349906,21
2,tool_to_tool__tool_use,3,0.575,0.345569,34
3,tool_to_tool__tool_use,4,0.475,0.339990,19
4,tool_to_tool__tool_use,5,0.575,0.370904,34
5,tool_to_assistant__grounding,1,0.575,0.212577,9
6,tool_to_assistant__grounding,2,0.475,0.212393,14
7,tool_to_assistant__grounding,3,0.575,0.219697,9
8,tool_to_assistant__grounding,4,0.575,0.227459,9
9,tool_to_assistant__grounding,5,0.575,0.216071,9


,mean_threshold,median_threshold,min_threshold,max_threshold,mean_eval_overrides
task,,,,,
assistant_to_tool__constraint,0.610,0.625,0.500,0.700,5.8
tool_to_assistant__grounding,0.555,0.575,0.475,0.575,10.0
tool_to_tool__tool_use,0.560,0.575,0.475,0.700,23.0


In [70]:
# ============================================================
# 45. End-to-end performance of each specialist
# ============================================================

specialist_global_predictions = {}

rows = []

for task, (
    transition,
    target_family,
) in refinement_tasks.items():

    idx, y_bin = refinement_indices[
        task
    ]

    selected_local = (
        specialist_crossfit_choice[
            task
        ]
    )

    selected_idx = idx[
        selected_local
    ]

    target_id = class_to_idx[
        target_family
    ]

    pred = (
        base_global_pred.copy()
    )

    pred[
        selected_idx
    ] = target_id

    specialist_global_predictions[
        task
    ] = pred

    correct = (
        pred == y_train
    )

    rescues = (
        (~base_global_correct)
        &
        correct
    )

    breaks = (
        base_global_correct
        &
        (~correct)
    )

    good = int(
        (
            y_train[
                selected_idx
            ]
            == target_id
        ).sum()
    )

    overrides = len(
        selected_idx
    )

    rows.append({
        "model":
            task,

        "overrides":
            overrides,

        "good_overrides":
            good,

        "bad_overrides":
            overrides - good,

        "override_precision":
            (
                good / overrides
                if overrides > 0
                else np.nan
            ),

        "rescues":
            int(
                rescues.sum()
            ),

        "breaks":
            int(
                breaks.sum()
            ),

        "net":
            int(
                rescues.sum()
                -
                breaks.sum()
            ),

        **metrics_local(pred),
    })


specialist_end_to_end_df = pd.DataFrame(
    rows
)

display(
    specialist_end_to_end_df
    .sort_values(
        "macro_f1",
        ascending=False,
    )
    .round(4)
)

,model,overrides,good_overrides,bad_overrides,override_precision,rescues,breaks,net,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,tool_to_tool__tool_use,115,46,69,0.4000,46,45,1,0.5158,0.4607,0.4726,0.5076
1,tool_to_assistant__grounding,50,20,30,0.4000,20,14,6,0.5191,0.4477,0.4650,0.5059
2,assistant_to_tool__constraint,29,12,17,0.4138,12,12,0,0.5151,0.4394,0.4578,0.4992


In [71]:
# ============================================================
# 46. Combined specialized refinement architecture
# ============================================================

combined_pred = (
    base_global_pred.copy()
)

combined_override_rows = []

for task, (
    transition,
    target_family,
) in refinement_tasks.items():

    idx, y_bin = refinement_indices[
        task
    ]

    selected_local = (
        specialist_crossfit_choice[
            task
        ]
    )

    selected_idx = idx[
        selected_local
    ]

    target_id = class_to_idx[
        target_family
    ]

    combined_pred[
        selected_idx
    ] = target_id

    good = (
        y_train[
            selected_idx
        ]
        == target_id
    )

    combined_override_rows.append({
        "task":
            task,

        "overrides":
            len(
                selected_idx
            ),

        "good":
            int(
                good.sum()
            ),

        "bad":
            int(
                (~good).sum()
            ),

        "precision":
            (
                good.mean()
                if len(good) > 0
                else np.nan
            ),
    })


combined_correct = (
    combined_pred == y_train
)

combined_rescues = (
    (~base_global_correct)
    &
    combined_correct
)

combined_breaks = (
    base_global_correct
    &
    (~combined_correct)
)

print(
    "BASE:",
    metrics_local(
        base_global_pred
    )
)

print(
    "\nCOMBINED SPECIALISTS:",
    metrics_local(
        combined_pred
    )
)

print(
    "\nRescues:",
    combined_rescues.sum()
)

print(
    "Breaks:",
    combined_breaks.sum()
)

print(
    "Net:",
    combined_rescues.sum()
    -
    combined_breaks.sum()
)

display(
    pd.DataFrame(
        combined_override_rows
    ).round(4)
)

BASE: {'accuracy': 0.5151108126259234, 'balanced_accuracy': 0.43550722847563206, 'macro_f1': 0.45483336181761536, 'weighted_f1': 0.4969282458539297}

COMBINED SPECIALISTS: {'accuracy': 0.5198119543317663, 'balanced_accuracy': 0.47677506290225635, 'macro_f1': 0.48550123369725373, 'weighted_f1': 0.5183667474614619}

Rescues: 78
Breaks: 71
Net: 7


,task,overrides,good,bad,precision
0,tool_to_tool__tool_use,115,46,69,0.4000
1,tool_to_assistant__grounding,50,20,30,0.4000
2,assistant_to_tool__constraint,29,12,17,0.4138


In [72]:
# ============================================================
# 47. Current architecture comparison
# ============================================================

comparison_rows = [
    {
        "model":
            "no_t1_base",
        **metrics_local(
            remove_t1_pred
        ),
    },
    {
        "model":
            "full_t1",
        **metrics_local(
            full_pred
        ),
    },
    {
        "model":
            "uncertainty_gated_t1",
        **metrics_local(
            crossfit_uncertainty_pred
        ),
    },
    {
        "model":
            "specialized_refinement",
        **metrics_local(
            combined_pred
        ),
    },
]

architecture_comparison_df = (
    pd.DataFrame(
        comparison_rows
    )
    .sort_values(
        [
            "macro_f1",
            "accuracy",
        ],
        ascending=False,
    )
)

display(
    architecture_comparison_df.round(4)
)

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
2,uncertainty_gated_t1,0.5406,0.4985,0.5105,0.5368
1,full_t1,0.5379,0.4983,0.5105,0.5352
3,specialized_refinement,0.5198,0.4768,0.4855,0.5184
0,no_t1_base,0.5151,0.4355,0.4548,0.4969


Your ranking is now:

Model	Accuracy	Balanced Acc.	Macro-F1
No t-1	.5151	.4355	.4548
Specialist refinement	.5198	.4768	.4855
Full t-1	.5379	.4983	.5105
Uncertainty-gated t-1	.5406	.4985	.5105

So I would not abandon the specialists. Instead, change their role.

The current experiment asked:

“When semantic-only predicts workflow, can a specialist replace workflow with a specific family?”

But your previous experiments already showed that t-1 is doing something much stronger: 159/181 changed predictions move OUT of workflow.

So the better architecture is:

semantic model → uncertainty gate → t-1 trajectory model → specialist verifier

The specialist should decide whether to accept/reject the trajectory model's proposed refinement, rather than independently generate the refinement.

That is a much easier binary problem.

Also notice the specialist ranking:

Tool→Tool tool-use: ROC-AUC .8243
Tool→Assistant grounding: ROC-AUC .7156
Assistant→Tool constraint: ROC-AUC .7177

There is real ranking signal. The problem is that thresholded override precision is only ~40%, so we're using that signal too aggressively.

The next experiment should therefore ask:

Can we predict when the trajectory/t-1 refinement is trustworthy?

This directly attacks the 44 breaks in your best t-1 gate.

In [74]:
# ============================================================
# 48. Build trajectory refinement / intervention dataset
# FIXED: no dependency on `le`
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Recover class ordering used by the probability matrices
# ------------------------------------------------------------
# IMPORTANT:
# This assumes the original multiclass model used sorted
# failure-family labels, as sklearn LabelEncoder normally does.
# ------------------------------------------------------------

class_names = np.sort(
    train_targets["failure_family"]
    .dropna()
    .unique()
)

print("Class order:", class_names)
print("Number of classes:", len(class_names))
print("Probability columns:", full_prob.shape[1])

assert len(class_names) == full_prob.shape[1], (
    f"Class mismatch: {len(class_names)} labels "
    f"but {full_prob.shape[1]} probability columns"
)

def decode_predictions(pred):
    pred = np.asarray(pred, dtype=int)
    return class_names[pred]


# ------------------------------------------------------------
# Make sure prediction arrays exist
# ------------------------------------------------------------

full_pred = np.argmax(
    full_prob,
    axis=1
)

remove_t1_pred = np.argmax(
    remove_t1_prob,
    axis=1
)

semantic_pred = remove_t1_pred
trajectory_pred = full_pred


# ------------------------------------------------------------
# Rows where t-1 changes the prediction
# ------------------------------------------------------------

changed_mask = (
    semantic_pred != trajectory_pred
)

changed_idx = np.where(
    changed_mask
)[0]

print(
    "Trajectory interventions:",
    len(changed_idx)
)


# ------------------------------------------------------------
# Encode true labels using the SAME class ordering
# ------------------------------------------------------------

class_to_idx = {
    name: i
    for i, name in enumerate(class_names)
}

y_train_local = (
    train_targets["failure_family"]
    .map(class_to_idx)
    .to_numpy()
)

assert not pd.isna(y_train_local).any()

y_train_local = y_train_local.astype(int)


# ------------------------------------------------------------
# Compare semantic vs trajectory correctness
# ------------------------------------------------------------

semantic_correct = (
    semantic_pred == y_train_local
)

trajectory_correct = (
    trajectory_pred == y_train_local
)


accept_rescue = (
    (~semantic_correct)
    &
    trajectory_correct
)

accept_break = (
    semantic_correct
    &
    (~trajectory_correct)
)

accept_wrong_to_wrong = (
    (~semantic_correct)
    &
    (~trajectory_correct)
)


changed_effect = np.full(
    len(y_train_local),
    "unchanged",
    dtype=object,
)

changed_effect[
    accept_rescue
] = "rescue"

changed_effect[
    accept_break
] = "break"

changed_effect[
    accept_wrong_to_wrong
] = "wrong_to_wrong"


# ------------------------------------------------------------
# Build intervention dataframe
# ------------------------------------------------------------

intervention_df = pd.DataFrame({

    "idx":
        changed_idx,

    "transition_type":
        transition_structure_df.loc[
            changed_idx,
            "transition_type"
        ].to_numpy(),

    "true_family":
        train_targets.loc[
            changed_idx,
            "failure_family"
        ].to_numpy(),

    "semantic_prediction":
        decode_predictions(
            semantic_pred[
                changed_idx
            ]
        ),

    "trajectory_prediction":
        decode_predictions(
            trajectory_pred[
                changed_idx
            ]
        ),

    "effect":
        changed_effect[
            changed_idx
        ],

    "trajectory_correct":
        trajectory_correct[
            changed_idx
        ].astype(int),

    "semantic_correct":
        semantic_correct[
            changed_idx
        ].astype(int),
})


print("\nEffect distribution:")
print(
    intervention_df[
        "effect"
    ].value_counts()
)

print("\nPrediction transitions:")
print(
    intervention_df[
        [
            "semantic_prediction",
            "trajectory_prediction"
        ]
    ]
    .value_counts()
    .head(15)
)

display(
    intervention_df.head(10)
)

Class order: ['constraint_error' 'grounding_state_error' 'reasoning_value_error'
 'tool_use_error' 'workflow_error']
Number of classes: 5
Probability columns: 5
Trajectory interventions: 181

Effect distribution:
effect
wrong_to_wrong    126
break              41
rescue             14
Name: count, dtype: int64

Prediction transitions:
semantic_prediction    trajectory_prediction
constraint_error       reasoning_value_error    67
                       tool_use_error           50
                       grounding_state_error    41
grounding_state_error  tool_use_error           18
constraint_error       workflow_error            1
grounding_state_error  reasoning_value_error     1
                       workflow_error            1
reasoning_value_error  grounding_state_error     1
workflow_error         tool_use_error            1
Name: count, dtype: int64


,idx,transition_type,true_family,semantic_prediction,trajectory_prediction,effect,trajectory_correct,semantic_correct
0,12,TOOL_CALL->ASSISTANT,workflow_error,constraint_error,tool_use_error,wrong_to_wrong,0,0
1,21,TOOL_CALL->ASSISTANT,grounding_state_error,constraint_error,tool_use_error,wrong_to_wrong,0,0
2,30,ASSISTANT->TOOL_CALL,tool_use_error,constraint_error,reasoning_value_error,wrong_to_wrong,0,0
3,34,TOOL_CALL->ASSISTANT,workflow_error,constraint_error,tool_use_error,wrong_to_wrong,0,0
4,79,TOOL_CALL->ASSISTANT,grounding_state_error,constraint_error,tool_use_error,wrong_to_wrong,0,0
5,91,TOOL_CALL->ASSISTANT,grounding_state_error,grounding_state_error,tool_use_error,break,0,1
6,93,TOOL_CALL->ASSISTANT,grounding_state_error,constraint_error,tool_use_error,wrong_to_wrong,0,0
7,109,TOOL_CALL->ASSISTANT,constraint_error,constraint_error,grounding_state_error,break,0,1
8,120,ASSISTANT->TOOL_CALL,tool_use_error,constraint_error,reasoning_value_error,wrong_to_wrong,0,0
9,124,ASSISTANT->TOOL_CALL,tool_use_error,constraint_error,reasoning_value_error,wrong_to_wrong,0,0


In [76]:
# ============================================================
# 48A. Recover the ACTUAL probability-column class mapping
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

families = (
    train_targets["failure_family"]
    .dropna()
    .unique()
    .tolist()
)

print("Families:", families)

# Raw probability-column predictions
semantic_idx = np.argmax(remove_t1_prob, axis=1)
trajectory_idx = np.argmax(full_prob, axis=1)

# ------------------------------------------------------------
# Inspect which TRUE families are associated with each
# probability-column argmax.
#
# This does NOT by itself prove the mapping, but gives us
# a diagnostic view.
# ------------------------------------------------------------

mapping_diag = pd.crosstab(
    semantic_idx,
    train_targets["failure_family"],
    normalize="index"
)

print("\nSemantic argmax column -> true-family distribution")
display(mapping_diag.round(3))

print("\nCounts")
display(
    pd.crosstab(
        semantic_idx,
        train_targets["failure_family"]
    )
)

# ------------------------------------------------------------
# Most important diagnostic:
# frequency of raw prediction indices
# ------------------------------------------------------------

print("\nSemantic prediction-index counts:")
print(
    pd.Series(semantic_idx)
    .value_counts()
    .sort_index()
)

print("\nTrajectory prediction-index counts:")
print(
    pd.Series(trajectory_idx)
    .value_counts()
    .sort_index()
)

# Changed rows only
changed_idx = np.where(
    semantic_idx != trajectory_idx
)[0]

print("\nChanged rows:", len(changed_idx))

print("\nSemantic indices among changed:")
print(
    pd.Series(
        semantic_idx[changed_idx]
    )
    .value_counts()
    .sort_index()
)

print("\nTrajectory indices among changed:")
print(
    pd.Series(
        trajectory_idx[changed_idx]
    )
    .value_counts()
    .sort_index()
)

Families: ['constraint_error', 'tool_use_error', 'workflow_error', 'grounding_state_error', 'reasoning_value_error']

Semantic argmax column -> true-family distribution


failure_family,constraint_error,grounding_state_error,reasoning_value_error,tool_use_error,workflow_error
row_0,,,,,
0,0.167,0.123,0.007,0.150,0.553
1,0.422,0.175,0.023,0.083,0.297
2,0.091,0.053,0.015,0.523,0.318
3,0.194,0.471,0.013,0.058,0.265
4,0.038,0.154,0.538,0.115,0.154



Counts


failure_family,constraint_error,grounding_state_error,reasoning_value_error,tool_use_error,workflow_error
row_0,,,,,
0,146,107,6,131,483
1,128,53,7,25,90
2,12,7,2,69,42
3,30,73,2,9,41
4,1,4,14,3,4



Semantic prediction-index counts:
0    873
1    303
2    132
3    155
4     26
Name: count, dtype: int64

Trajectory prediction-index counts:
0    714
1    325
2    199
3    224
4     27
Name: count, dtype: int64

Changed rows: 181

Semantic indices among changed:
0    159
1     20
2      1
4      1
Name: count, dtype: int64

Trajectory indices among changed:
1    42
2    68
3    69
4     2
Name: count, dtype: int64


In [77]:
# ============================================================
# 48B. Search notebook namespace for original class metadata
# ============================================================

candidate_names = [
    "classes",
    "classes_",
    "class_names",
    "labels",
    "label_names",
    "family_names",
    "failure_families",
    "target_classes",
    "class_order",
    "label_encoder",
    "encoder",
    "le",
    "clf",
    "model",
]

for name in candidate_names:
    if name not in globals():
        continue

    obj = globals()[name]

    print("\n" + "=" * 70)
    print(name, type(obj))

    try:
        print("value:", obj)
    except Exception:
        pass

    try:
        print("classes_:", obj.classes_)
    except Exception:
        pass


class_names <class 'numpy.ndarray'>
value: ['constraint_error' 'grounding_state_error' 'reasoning_value_error'
 'tool_use_error' 'workflow_error']

encoder <class 'sentence_transformers.SentenceTransformer.SentenceTransformer'>
value: SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

model <class 'sklearn.linear_model._logistic.LogisticRegression'>
value: LogisticRegression(C=0.1, class_weight='balanced', max_iter=5000,
                   random_state=42)
classes_: [0 1]


In [78]:
# Snapshot globals first to avoid:
# RuntimeError: dictionary changed size during iteration

namespace_snapshot = list(globals().items())

found = []

for name, obj in namespace_snapshot:

    if name.startswith("_"):
        continue

    try:
        classes = getattr(obj, "classes_", None)

        if classes is not None:
            found.append(
                (
                    name,
                    type(obj).__name__,
                    classes
                )
            )

    except Exception:
        pass

print("Objects containing classes_:")

for name, obj_type, classes in found:
    print(
        f"\n{name} [{obj_type}]"
    )
    print(classes)

Objects containing classes_:

model [LogisticRegression]
[0 1]

obj [LogisticRegression]
[0 1]


In [79]:
# ============================================================
# 48C. Canonical class mapping for Notebook 16
# ============================================================

CLASS_NAMES = np.array([
    "workflow_error",
    "constraint_error",
    "tool_use_error",
    "grounding_state_error",
    "reasoning_value_error",
])

class_to_idx = {
    name: i
    for i, name in enumerate(CLASS_NAMES)
}

def decode_predictions(pred):
    pred = np.asarray(pred, dtype=int)
    return CLASS_NAMES[pred]


print("Canonical probability-column mapping:")

for i, name in enumerate(CLASS_NAMES):
    print(i, "->", name)

Canonical probability-column mapping:
0 -> workflow_error
1 -> constraint_error
2 -> tool_use_error
3 -> grounding_state_error
4 -> reasoning_value_error


In [80]:
# ============================================================
# 48D. Rebuild trajectory intervention dataset correctly
# ============================================================

semantic_pred = np.argmax(
    remove_t1_prob,
    axis=1,
)

trajectory_pred = np.argmax(
    full_prob,
    axis=1,
)

# True labels in the SAME index space
y_train_canonical = (
    train_targets["failure_family"]
    .map(class_to_idx)
    .to_numpy()
)

assert not pd.isna(
    y_train_canonical
).any()

y_train_canonical = (
    y_train_canonical.astype(int)
)

# ------------------------------------------------------------
# Prediction changes
# ------------------------------------------------------------

changed_mask = (
    semantic_pred
    != trajectory_pred
)

changed_idx = np.where(
    changed_mask
)[0]

semantic_correct = (
    semantic_pred
    == y_train_canonical
)

trajectory_correct = (
    trajectory_pred
    == y_train_canonical
)

rescue_mask = (
    (~semantic_correct)
    &
    trajectory_correct
)

break_mask = (
    semantic_correct
    &
    (~trajectory_correct)
)

wrong_to_wrong_mask = (
    (~semantic_correct)
    &
    (~trajectory_correct)
    &
    changed_mask
)

changed_effect = np.full(
    len(y_train_canonical),
    "unchanged",
    dtype=object,
)

changed_effect[
    rescue_mask
] = "rescue"

changed_effect[
    break_mask
] = "break"

changed_effect[
    wrong_to_wrong_mask
] = "wrong_to_wrong"

# ------------------------------------------------------------
# Intervention dataframe
# ------------------------------------------------------------

intervention_df = pd.DataFrame({
    "idx":
        changed_idx,

    "transition_type":
        transition_structure_df.loc[
            changed_idx,
            "transition_type"
        ].to_numpy(),

    "true_family":
        train_targets.loc[
            changed_idx,
            "failure_family"
        ].to_numpy(),

    "semantic_prediction":
        decode_predictions(
            semantic_pred[
                changed_idx
            ]
        ),

    "trajectory_prediction":
        decode_predictions(
            trajectory_pred[
                changed_idx
            ]
        ),

    "effect":
        changed_effect[
            changed_idx
        ],

    "trajectory_correct":
        trajectory_correct[
            changed_idx
        ].astype(int),

    "semantic_correct":
        semantic_correct[
            changed_idx
        ].astype(int),
})


print(
    "Trajectory interventions:",
    len(intervention_df)
)

print("\nEffect distribution:")
print(
    intervention_df[
        "effect"
    ].value_counts()
)

print("\nSemantic predictions:")
print(
    intervention_df[
        "semantic_prediction"
    ].value_counts()
)

print("\nTrajectory predictions:")
print(
    intervention_df[
        "trajectory_prediction"
    ].value_counts()
)

Trajectory interventions: 181

Effect distribution:
effect
rescue            87
break             53
wrong_to_wrong    41
Name: count, dtype: int64

Semantic predictions:
semantic_prediction
workflow_error           159
constraint_error          20
reasoning_value_error      1
tool_use_error             1
Name: count, dtype: int64

Trajectory predictions:
trajectory_prediction
grounding_state_error    69
tool_use_error           68
constraint_error         42
reasoning_value_error     2
Name: count, dtype: int64


In [81]:
# ============================================================
# 49. Intervention targets
# ============================================================

intervention_df[
    "accept_is_correct"
] = (
    intervention_df[
        "trajectory_correct"
    ]
    .astype(int)
)

intervention_df[
    "accept_is_beneficial"
] = (
    intervention_df[
        "effect"
    ]
    == "rescue"
).astype(int)

intervention_df[
    "accept_is_harmful"
] = (
    intervention_df[
        "effect"
    ]
    == "break"
).astype(int)


display(
    intervention_df[
        [
            "effect",
            "accept_is_correct",
            "accept_is_beneficial",
            "accept_is_harmful",
        ]
    ]
    .value_counts()
    .reset_index(
        name="count"
    )
)

,effect,accept_is_correct,accept_is_beneficial,accept_is_harmful,count
0,rescue,1,1,0,87
1,break,0,0,1,53
2,wrong_to_wrong,0,0,0,41


In [82]:
# ============================================================
# 50. Intervention-verifier features
# ============================================================

idx = changed_idx

semantic_prob = (
    remove_t1_prob[idx]
)

trajectory_prob = (
    full_prob[idx]
)

# ------------------------------------------------------------
# Confidence / margin / entropy
# ------------------------------------------------------------

semantic_conf = (
    semantic_prob.max(axis=1)
)

trajectory_conf = (
    trajectory_prob.max(axis=1)
)

semantic_sorted = np.sort(
    semantic_prob,
    axis=1,
)

trajectory_sorted = np.sort(
    trajectory_prob,
    axis=1,
)

semantic_margin = (
    semantic_sorted[:, -1]
    -
    semantic_sorted[:, -2]
)

trajectory_margin = (
    trajectory_sorted[:, -1]
    -
    trajectory_sorted[:, -2]
)

semantic_entropy = -np.sum(
    semantic_prob
    *
    np.log(
        semantic_prob + 1e-12
    ),
    axis=1,
)

trajectory_entropy = -np.sum(
    trajectory_prob
    *
    np.log(
        trajectory_prob + 1e-12
    ),
    axis=1,
)

# ------------------------------------------------------------
# Probability assigned to proposed trajectory class
# ------------------------------------------------------------

proposed_class = (
    trajectory_pred[idx]
)

row_idx = np.arange(
    len(idx)
)

proposed_prob_semantic = (
    semantic_prob[
        row_idx,
        proposed_class
    ]
)

proposed_prob_trajectory = (
    trajectory_prob[
        row_idx,
        proposed_class
    ]
)

proposed_prob_gain = (
    proposed_prob_trajectory
    -
    proposed_prob_semantic
)

# Probability of class being abandoned
semantic_class = (
    semantic_pred[idx]
)

abandoned_prob_semantic = (
    semantic_prob[
        row_idx,
        semantic_class
    ]
)

abandoned_prob_trajectory = (
    trajectory_prob[
        row_idx,
        semantic_class
    ]
)

abandoned_prob_drop = (
    abandoned_prob_semantic
    -
    abandoned_prob_trajectory
)

# ------------------------------------------------------------
# Distribution shift
# ------------------------------------------------------------

prob_l1_shift = np.sum(
    np.abs(
        trajectory_prob
        -
        semantic_prob
    ),
    axis=1,
)

prob_l2_shift = np.linalg.norm(
    trajectory_prob
    -
    semantic_prob,
    axis=1,
)

# ------------------------------------------------------------
# Raw t-1 geometry
# ------------------------------------------------------------

t1_l1 = distance_df.loc[
    idx,
    "current_tminus1_l1"
].to_numpy()

t1_l2 = distance_df.loc[
    idx,
    "current_tminus1_l2"
].to_numpy()

# ------------------------------------------------------------
# DataFrame
# ------------------------------------------------------------

intervention_feature_df = pd.DataFrame({
    "semantic_confidence":
        semantic_conf,

    "semantic_margin":
        semantic_margin,

    "semantic_entropy":
        semantic_entropy,

    "trajectory_confidence":
        trajectory_conf,

    "trajectory_margin":
        trajectory_margin,

    "trajectory_entropy":
        trajectory_entropy,

    "confidence_gain":
        trajectory_conf
        - semantic_conf,

    "margin_gain":
        trajectory_margin
        - semantic_margin,

    "entropy_drop":
        semantic_entropy
        - trajectory_entropy,

    "proposed_prob_semantic":
        proposed_prob_semantic,

    "proposed_prob_trajectory":
        proposed_prob_trajectory,

    "proposed_prob_gain":
        proposed_prob_gain,

    "abandoned_prob_semantic":
        abandoned_prob_semantic,

    "abandoned_prob_trajectory":
        abandoned_prob_trajectory,

    "abandoned_prob_drop":
        abandoned_prob_drop,

    "prob_l1_shift":
        prob_l1_shift,

    "prob_l2_shift":
        prob_l2_shift,

    "t1_l1":
        t1_l1,

    "t1_l2":
        t1_l2,
})

print(
    intervention_feature_df.shape
)

display(
    intervention_feature_df
    .describe()
    .T
    .round(4)
)

(181, 19)


,count,mean,std,min,25%,50%,75%,max
semantic_confidence,181.0,0.4524,0.0712,0.2613,0.3949,0.4548,0.5065,0.6423
semantic_margin,181.0,0.0966,0.0696,0.0011,0.0381,0.0874,0.1350,0.3206
semantic_entropy,181.0,1.1276,0.1791,0.7527,0.9486,1.1485,1.2786,1.5188
trajectory_confidence,181.0,0.4437,0.0808,0.2733,0.3807,0.4434,0.5081,0.6145
trajectory_margin,181.0,0.0982,0.0753,0.0012,0.0382,0.0815,0.1458,0.3245
trajectory_entropy,181.0,1.1572,0.1804,0.7840,0.9856,1.2080,1.3054,1.4742
confidence_gain,181.0,-0.0088,0.0664,-0.1583,-0.0576,-0.0141,0.0372,0.1560
margin_gain,181.0,0.0016,0.1169,-0.2712,-0.0808,0.0071,0.0847,0.2912
entropy_drop,181.0,-0.0295,0.0517,-0.1530,-0.0655,-0.0250,0.0042,0.0927
proposed_prob_semantic,181.0,0.3523,0.0689,0.1730,0.3011,0.3519,0.4080,0.4821


In [83]:
# ============================================================
# 51. Structural intervention features
# ============================================================

intervention_feature_df[
    "transition_type"
] = (
    transition_structure_df.loc[
        idx,
        "transition_type"
    ].to_numpy()
)

intervention_feature_df[
    "semantic_family"
] = decode_predictions(
    semantic_pred[idx]
)

intervention_feature_df[
    "trajectory_family"
] = decode_predictions(
    trajectory_pred[idx]
)

intervention_feature_df[
    "prediction_transition"
] = (
    intervention_feature_df[
        "semantic_family"
    ]
    +
    "->"
    +
    intervention_feature_df[
        "trajectory_family"
    ]
)

display(
    intervention_feature_df[
        [
            "transition_type",
            "semantic_family",
            "trajectory_family",
            "prediction_transition",
        ]
    ]
    .value_counts()
    .reset_index(
        name="count"
    )
    .head(30)
)

,transition_type,semantic_family,trajectory_family,prediction_transition,count
0,TOOL_CALL->TOOL_CALL,workflow_error,tool_use_error,workflow_error->tool_use_error,39
1,TOOL_CALL->ASSISTANT,workflow_error,grounding_state_error,workflow_error->grounding_state_error,19
2,ASSISTANT->TOOL_CALL,workflow_error,tool_use_error,workflow_error->tool_use_error,18
3,TOOL_CALL->ASSISTANT,workflow_error,constraint_error,workflow_error->constraint_error,15
4,ASSISTANT->TOOL_CALL,workflow_error,constraint_error,workflow_error->constraint_error,15
5,TOOL_CALL->TOOL_CALL,workflow_error,grounding_state_error,workflow_error->grounding_state_error,12
6,ASSISTANT->TOOL_CALL,workflow_error,grounding_state_error,workflow_error->grounding_state_error,11
7,ASSISTANT->ASSISTANT,workflow_error,constraint_error,workflow_error->constraint_error,9
8,TOOL_CALL->ASSISTANT,workflow_error,tool_use_error,workflow_error->tool_use_error,9
9,TOOL_CALL->ASSISTANT,constraint_error,grounding_state_error,constraint_error->grounding_state_error,8


In [84]:
# ============================================================
# 52. Univariate rescue-vs-break diagnostics
# ============================================================

from sklearn.metrics import roc_auc_score

# Restrict to cases where accepting trajectory has a clear
# binary consequence: rescue or break.
rb_mask = intervention_df[
    "effect"
].isin(
    ["rescue", "break"]
).to_numpy()

y_rb = (
    intervention_df.loc[
        rb_mask,
        "effect"
    ]
    == "rescue"
).astype(int).to_numpy()

print("Rescue/break examples:", rb_mask.sum())
print("Rescues:", y_rb.sum())
print("Breaks:", len(y_rb) - y_rb.sum())
print("Rescue prevalence:", y_rb.mean())


numeric_cols = (
    intervention_feature_df
    .select_dtypes(include=np.number)
    .columns
)

rows = []

for col in numeric_cols:

    values = (
        intervention_feature_df.loc[
            rb_mask,
            col
        ]
        .to_numpy()
    )

    if np.nanstd(values) < 1e-12:
        continue

    valid = np.isfinite(values)

    if valid.sum() < 10:
        continue

    auc = roc_auc_score(
        y_rb[valid],
        values[valid],
    )

    rows.append({
        "feature": col,
        "auc": auc,
        "direction_free_auc": max(
            auc,
            1 - auc
        ),

        "rescue_mean":
            values[
                valid
                &
                (y_rb == 1)
            ].mean(),

        "break_mean":
            values[
                valid
                &
                (y_rb == 0)
            ].mean(),
    })


univariate_intervention_df = (
    pd.DataFrame(rows)
    .sort_values(
        "direction_free_auc",
        ascending=False
    )
)

display(
    univariate_intervention_df.round(4)
)

Rescue/break examples: 140
Rescues: 87
Breaks: 53
Rescue prevalence: 0.6214285714285714


,feature,auc,direction_free_auc,rescue_mean,break_mean
6,confidence_gain,0.6255,0.6255,0.0054,-0.0249
4,trajectory_margin,0.6183,0.6183,0.1162,0.0866
7,margin_gain,0.6137,0.6137,0.0225,-0.0236
8,entropy_drop,0.5964,0.5964,-0.0228,-0.0391
3,trajectory_confidence,0.5702,0.5702,0.4647,0.4447
10,proposed_prob_trajectory,0.5702,0.5702,0.4647,0.4447
11,proposed_prob_gain,0.5689,0.5689,0.1009,0.0887
0,semantic_confidence,0.4540,0.5460,0.4593,0.4696
12,abandoned_prob_semantic,0.4540,0.5460,0.4593,0.4696
1,semantic_margin,0.4620,0.5380,0.0937,0.1102


In [85]:
# ============================================================
# 53. Reliability by transition × proposed prediction move
# ============================================================

analysis_df = pd.concat(
    [
        intervention_df.reset_index(drop=True),

        intervention_feature_df[
            [
                "prediction_transition",
                "semantic_confidence",
                "semantic_margin",
                "semantic_entropy",
                "trajectory_confidence",
                "trajectory_margin",
                "trajectory_entropy",
                "proposed_prob_gain",
                "abandoned_prob_drop",
                "prob_l1_shift",
                "prob_l2_shift",
                "t1_l1",
                "t1_l2",
            ]
        ].reset_index(drop=True),
    ],
    axis=1,
)

transition_reliability = (
    analysis_df
    .groupby(
        [
            "transition_type",
            "prediction_transition",
        ]
    )
    .agg(
        support=("effect", "size"),

        rescues=(
            "effect",
            lambda x: (x == "rescue").sum()
        ),

        breaks=(
            "effect",
            lambda x: (x == "break").sum()
        ),

        wrong_to_wrong=(
            "effect",
            lambda x: (
                x == "wrong_to_wrong"
            ).sum()
        ),

        mean_semantic_confidence=(
            "semantic_confidence",
            "mean"
        ),

        mean_trajectory_confidence=(
            "trajectory_confidence",
            "mean"
        ),

        mean_proposed_gain=(
            "proposed_prob_gain",
            "mean"
        ),

        mean_abandoned_drop=(
            "abandoned_prob_drop",
            "mean"
        ),

        mean_prob_shift=(
            "prob_l1_shift",
            "mean"
        ),
    )
    .reset_index()
)

transition_reliability["net"] = (
    transition_reliability["rescues"]
    -
    transition_reliability["breaks"]
)

transition_reliability["rescue_rate"] = (
    transition_reliability["rescues"]
    /
    transition_reliability["support"]
)

transition_reliability["break_rate"] = (
    transition_reliability["breaks"]
    /
    transition_reliability["support"]
)

transition_reliability["accept_accuracy"] = (
    transition_reliability["rescues"]
    /
    (
        transition_reliability["rescues"]
        +
        transition_reliability["breaks"]
        +
        transition_reliability["wrong_to_wrong"]
    )
)

display(
    transition_reliability
    .sort_values(
        ["net", "support"],
        ascending=[False, False]
    )
    .round(4)
)

,transition_type,prediction_transition,support,rescues,breaks,wrong_to_wrong,mean_semantic_confidence,mean_trajectory_confidence,mean_proposed_gain,mean_abandoned_drop,mean_prob_shift,net,rescue_rate,break_rate,accept_accuracy
20,TOOL_CALL->TOOL_CALL,workflow_error->tool_use_error,39,23,13,3,0.5080,0.5206,0.0971,0.1069,0.2172,10,0.5897,0.3333,0.5897
14,TOOL_CALL->ASSISTANT,workflow_error->grounding_state_error,19,13,3,3,0.4193,0.4273,0.1363,0.1365,0.3059,10,0.6842,0.1579,0.6842
19,TOOL_CALL->TOOL_CALL,workflow_error->grounding_state_error,12,7,2,3,0.4182,0.4376,0.1130,0.1145,0.2537,5,0.5833,0.1667,0.5833
8,ASSISTANT->TOOL_CALL,workflow_error->constraint_error,15,8,4,3,0.4621,0.4210,0.0333,0.1101,0.2213,4,0.5333,0.2667,0.5333
11,TOOL_CALL->ASSISTANT,constraint_error->grounding_state_error,8,5,1,2,0.4160,0.4751,0.1462,0.0776,0.3019,4,0.6250,0.1250,0.6250
17,TOOL_CALL->TOOL_CALL,constraint_error->grounding_state_error,3,3,0,0,0.4731,0.5059,0.1186,0.0926,0.2419,3,1.0000,0.0000,1.0000
10,ASSISTANT->TOOL_CALL,workflow_error->tool_use_error,18,8,7,3,0.4609,0.4312,0.1065,0.1430,0.3031,1,0.4444,0.3889,0.4444
3,ASSISTANT->ASSISTANT,workflow_error->constraint_error,9,3,2,4,0.4539,0.4230,0.0321,0.0868,0.1749,1,0.3333,0.2222,0.3333
16,TOOL_CALL->ASSISTANT,workflow_error->tool_use_error,9,3,2,4,0.4020,0.3987,0.0793,0.1054,0.2391,1,0.3333,0.2222,0.3333
18,TOOL_CALL->TOOL_CALL,workflow_error->constraint_error,2,1,0,1,0.3915,0.3652,0.0121,0.0765,0.1554,1,0.5000,0.0000,0.5000


In [86]:
# ============================================================
# 54. Reliability of the major trajectory intervention types
# ============================================================

major_moves = (
    analysis_df
    .groupby(
        "prediction_transition"
    )
    .agg(
        support=("effect", "size"),

        rescues=(
            "effect",
            lambda x: (x == "rescue").sum()
        ),

        breaks=(
            "effect",
            lambda x: (x == "break").sum()
        ),

        wrong_to_wrong=(
            "effect",
            lambda x: (
                x == "wrong_to_wrong"
            ).sum()
        ),
    )
    .reset_index()
)

major_moves["net"] = (
    major_moves["rescues"]
    -
    major_moves["breaks"]
)

major_moves["rescue_rate"] = (
    major_moves["rescues"]
    /
    major_moves["support"]
)

major_moves["break_rate"] = (
    major_moves["breaks"]
    /
    major_moves["support"]
)

display(
    major_moves
    .sort_values(
        ["net", "support"],
        ascending=[False, False]
    )
    .round(4)
)

,prediction_transition,support,rescues,breaks,wrong_to_wrong,net,rescue_rate,break_rate
8,workflow_error->tool_use_error,67,34,23,10,11,0.5075,0.3433
6,workflow_error->grounding_state_error,50,24,13,13,11,0.4800,0.2600
0,constraint_error->grounding_state_error,18,10,4,4,6,0.5556,0.2222
5,workflow_error->constraint_error,41,17,13,11,4,0.4146,0.3171
1,constraint_error->reasoning_value_error,1,1,0,0,1,1.0000,0.0000
7,workflow_error->reasoning_value_error,1,1,0,0,1,1.0000,0.0000
2,constraint_error->tool_use_error,1,0,0,1,0,0.0000,0.0000
3,reasoning_value_error->grounding_state_error,1,0,0,1,0,0.0000,0.0000
4,tool_use_error->constraint_error,1,0,0,1,0,0.0000,0.0000


In [87]:
# ============================================================
# 55. Dominant mechanism diagnostics
# ============================================================

dominant_rules = [
    (
        "TOOL_CALL->TOOL_CALL",
        "workflow_error->tool_use_error",
    ),
    (
        "TOOL_CALL->ASSISTANT",
        "workflow_error->grounding_state_error",
    ),
    (
        "ASSISTANT->TOOL_CALL",
        "workflow_error->constraint_error",
    ),
]

rows = []

for transition, move in dominant_rules:

    sub = analysis_df[
        (analysis_df["transition_type"] == transition)
        &
        (
            analysis_df["prediction_transition"]
            == move
        )
    ].copy()

    if len(sub) == 0:
        continue

    rows.append({
        "transition_type": transition,
        "prediction_transition": move,

        "support": len(sub),

        "rescues":
            (sub["effect"] == "rescue").sum(),

        "breaks":
            (sub["effect"] == "break").sum(),

        "wrong_to_wrong":
            (
                sub["effect"]
                == "wrong_to_wrong"
            ).sum(),

        "net":
            (
                (sub["effect"] == "rescue").sum()
                -
                (sub["effect"] == "break").sum()
            ),

        "mean_semantic_confidence":
            sub[
                "semantic_confidence"
            ].mean(),

        "mean_proposed_prob_gain":
            sub[
                "proposed_prob_gain"
            ].mean(),

        "mean_prob_l1_shift":
            sub[
                "prob_l1_shift"
            ].mean(),

        "mean_t1_l2":
            sub[
                "t1_l2"
            ].mean(),
    })


dominant_mechanism_df = (
    pd.DataFrame(rows)
)

display(
    dominant_mechanism_df.round(4)
)

,transition_type,prediction_transition,support,rescues,breaks,wrong_to_wrong,net,mean_semantic_confidence,mean_proposed_prob_gain,mean_prob_l1_shift,mean_t1_l2
0,TOOL_CALL->TOOL_CALL,workflow_error->tool_use_error,39,23,13,3,10,0.5080,0.0971,0.2172,0.8846
1,TOOL_CALL->ASSISTANT,workflow_error->grounding_state_error,19,13,3,3,10,0.4193,0.1363,0.3059,1.1758
2,ASSISTANT->TOOL_CALL,workflow_error->constraint_error,15,8,4,3,4,0.4621,0.0333,0.2213,1.1460


This is the key result we needed. The intervention verifier problem is real, but it is not dominated by raw trajectory distance. The strongest univariate signals are about whether the trajectory model becomes more decisive relative to the semantic model.

The clearest rescue-vs-break indicators are confidence_gain with AUC 0.626, trajectory_margin at 0.618, and margin_gain at 0.614. Raw t1_l1 and t1_l2 are essentially chance at 0.507 / 0.506. So the useful question is no longer “how far did the representation move?” but rather:

Did adding trajectory evidence produce a more coherent and confident alternative decision?

The structural moves are also very strong. The best recurring intervention types are:

TOOL_CALL→TOOL_CALL + workflow→tool_use: 23 rescues, 13 breaks, net +10
TOOL_CALL→ASSISTANT + workflow→grounding: 13 rescues, 3 breaks, net +10
TOOL_CALL→TOOL_CALL + workflow→grounding: 7 rescues, 2 breaks, net +5
ASSISTANT→TOOL_CALL + workflow→constraint: 8 rescues, 4 breaks, net +4

And the harmful moves are structurally different:

TOOL_CALL→ASSISTANT + workflow→constraint: net −2
ASSISTANT→TOOL_CALL + workflow→grounding: net −2
ASSISTANT→ASSISTANT + workflow→grounding: net −2

This strongly supports a transition-aware intervention verifier.

Next experiment: learned accept/reject verifier

Now train only on the 140 cases where trajectory either rescued or broke the semantic prediction. Ignore wrong_to_wrong for the first experiment, because the target is clean

In [88]:
# ============================================================
# 56. Rescue-vs-break verifier dataset
# ============================================================

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)

rb_idx_local = np.where(
    intervention_df["effect"].isin(["rescue", "break"]).to_numpy()
)[0]

verifier_df = intervention_feature_df.iloc[
    rb_idx_local
].copy()

verifier_df["transition_type"] = (
    intervention_df.iloc[
        rb_idx_local
    ]["transition_type"].to_numpy()
)

verifier_df["semantic_family"] = (
    intervention_df.iloc[
        rb_idx_local
    ]["semantic_prediction"].to_numpy()
)

verifier_df["trajectory_family"] = (
    intervention_df.iloc[
        rb_idx_local
    ]["trajectory_prediction"].to_numpy()
)

verifier_df["prediction_transition"] = (
    verifier_df["semantic_family"]
    + "->"
    + verifier_df["trajectory_family"]
)

y_verifier = (
    intervention_df.iloc[
        rb_idx_local
    ]["effect"]
    == "rescue"
).astype(int).to_numpy()

global_rb_idx = intervention_df.iloc[
    rb_idx_local
]["idx"].to_numpy()

groups_verifier = groups_train[
    global_rb_idx
]

print("Verifier rows:", len(verifier_df))
print("Rescues:", y_verifier.sum())
print("Breaks:", len(y_verifier) - y_verifier.sum())
print("Positive prevalence:", y_verifier.mean())

Verifier rows: 140
Rescues: 87
Breaks: 53
Positive prevalence: 0.6214285714285714


In [89]:
# ============================================================
# 57. Verifier feature sets
# ============================================================

VERIFIER_NUMERIC = [
    "semantic_confidence",
    "semantic_margin",
    "semantic_entropy",

    "trajectory_confidence",
    "trajectory_margin",
    "trajectory_entropy",

    "confidence_gain",
    "margin_gain",
    "entropy_drop",

    "proposed_prob_gain",
    "abandoned_prob_drop",

    "prob_l1_shift",
    "prob_l2_shift",
]

VERIFIER_CATEGORICAL = [
    "transition_type",
    "semantic_family",
    "trajectory_family",
]

print(
    "Numeric:",
    len(VERIFIER_NUMERIC),
    "Categorical:",
    len(VERIFIER_CATEGORICAL),
)

Numeric: 13 Categorical: 3


In [90]:
# ============================================================
# 58. Cross-fitted intervention verifier
# ============================================================

numeric_transformer = Pipeline([
    (
        "scale",
        StandardScaler(),
    ),
])

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore",
)

preprocessor = ColumnTransformer([
    (
        "num",
        numeric_transformer,
        VERIFIER_NUMERIC,
    ),
    (
        "cat",
        categorical_transformer,
        VERIFIER_CATEGORICAL,
    ),
])

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

verifier_oof_prob = np.zeros(
    len(verifier_df),
    dtype=float,
)

selected_Cs = []

C_grid = [
    0.01,
    0.03,
    0.1,
    0.3,
    1.0,
]

for fold, (tr, va) in enumerate(
    cv.split(
        verifier_df,
        y_verifier,
        groups=groups_verifier,
    ),
    start=1,
):

    best_C = None
    best_score = -np.inf

    # small inner search on training portion
    for C in C_grid:

        model = Pipeline([
            (
                "prep",
                preprocessor,
            ),
            (
                "clf",
                LogisticRegression(
                    C=C,
                    class_weight="balanced",
                    max_iter=5000,
                    random_state=42,
                ),
            ),
        ])

        model.fit(
            verifier_df.iloc[tr],
            y_verifier[tr],
        )

        p_tr = model.predict_proba(
            verifier_df.iloc[tr]
        )[:, 1]

        score = average_precision_score(
            y_verifier[tr],
            p_tr,
        )

        if score > best_score:
            best_score = score
            best_C = C

    selected_Cs.append(best_C)

    final_model = Pipeline([
        (
            "prep",
            preprocessor,
        ),
        (
            "clf",
            LogisticRegression(
                C=best_C,
                class_weight="balanced",
                max_iter=5000,
                random_state=42,
            ),
        ),
    ])

    final_model.fit(
        verifier_df.iloc[tr],
        y_verifier[tr],
    )

    verifier_oof_prob[
        va
    ] = final_model.predict_proba(
        verifier_df.iloc[va]
    )[:, 1]

    print(
        f"Fold {fold}: C={best_C}"
    )

print("\nSelected Cs:", selected_Cs)

print(
    "PR-AUC:",
    average_precision_score(
        y_verifier,
        verifier_oof_prob,
    )
)

print(
    "ROC-AUC:",
    roc_auc_score(
        y_verifier,
        verifier_oof_prob,
    )
)

Fold 1: C=1.0
Fold 2: C=1.0
Fold 3: C=1.0
Fold 4: C=1.0
Fold 5: C=1.0

Selected Cs: [1.0, 1.0, 1.0, 1.0, 1.0]
PR-AUC: 0.6239038078599701
ROC-AUC: 0.5133376707872479


In [91]:
# ============================================================
# 59. Diagnostic verifier threshold sweep
# ============================================================

thresholds = np.arange(
    0.30,
    0.91,
    0.025,
)

rows = []

# Semantic/no-t1 prediction is the fallback.
base_pred = remove_t1_pred.copy()

for threshold in thresholds:

    pred = base_pred.copy()

    # Only the 140 rescue/break intervention rows are currently
    # eligible for this diagnostic.
    accept_local = (
        verifier_oof_prob
        >= threshold
    )

    accept_global_idx = (
        global_rb_idx[
            accept_local
        ]
    )

    pred[
        accept_global_idx
    ] = full_pred[
        accept_global_idx
    ]

    correct = (
        pred == y_train_canonical
    )

    base_correct = (
        base_pred == y_train_canonical
    )

    rescues = (
        (~base_correct)
        &
        correct
    ).sum()

    breaks = (
        base_correct
        &
        (~correct)
    ).sum()

    rows.append({
        "threshold":
            threshold,

        "accepted":
            int(
                accept_local.sum()
            ),

        "coverage":
            accept_local.mean(),

        "accept_precision":
            (
                y_verifier[
                    accept_local
                ].mean()
                if accept_local.sum() > 0
                else np.nan
            ),

        "rescues":
            int(rescues),

        "breaks":
            int(breaks),

        "net":
            int(
                rescues
                -
                breaks
            ),

        "accuracy":
            accuracy_score(
                y_train_canonical,
                pred,
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_train_canonical,
                pred,
            ),

        "macro_f1":
            f1_score(
                y_train_canonical,
                pred,
                average="macro",
                zero_division=0,
            ),

        "weighted_f1":
            f1_score(
                y_train_canonical,
                pred,
                average="weighted",
                zero_division=0,
            ),
    })

verifier_threshold_df = (
    pd.DataFrame(rows)
    .sort_values(
        [
            "macro_f1",
            "accuracy",
        ],
        ascending=False,
    )
)

display(
    verifier_threshold_df
    .head(20)
    .round(4)
)

,threshold,accepted,coverage,accept_precision,rescues,breaks,net,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,0.300,128,0.9143,0.6172,79,49,30,0.5353,0.4937,0.5086,0.5305
1,0.325,125,0.8929,0.6080,76,49,27,0.5332,0.4911,0.5061,0.5282
2,0.350,121,0.8643,0.6116,74,47,27,0.5332,0.4901,0.5054,0.5279
3,0.375,113,0.8071,0.6283,71,42,29,0.5346,0.4894,0.5052,0.5286
4,0.400,108,0.7714,0.6296,68,40,28,0.5339,0.4879,0.5040,0.5276
5,0.425,100,0.7143,0.6300,63,37,26,0.5326,0.4852,0.5016,0.5256
6,0.450,92,0.6571,0.6413,59,33,26,0.5326,0.4780,0.4961,0.5250
7,0.475,87,0.6214,0.6437,56,31,25,0.5319,0.4761,0.4943,0.5238
8,0.500,78,0.5571,0.6282,49,29,20,0.5285,0.4709,0.4889,0.5194
9,0.525,70,0.5000,0.6286,44,26,18,0.5272,0.4676,0.4857,0.5172


The verifier did not learn a meaningful rescue-vs-break ranking. ROC-AUC is only 0.513, essentially chance. PR-AUC 0.624 looks superficially decent, but the positive prevalence is already 0.621, so it is not adding useful discrimination. The threshold table confirms that: precision stays around the base rescue prevalence and does not improve systematically as the threshold rises.

So the conclusion is:

A generic learned verifier over the current numeric + structural features cannot reliably distinguish trajectory rescues from breaks.

The best thresholded result, around 0.30, gives accuracy 0.5353, still below the full t−1 model at 0.5379 and clearly below the uncertainty-gated model at 0.5406.

This means the next research step should not be more tuning of this verifier. We have evidence that the useful structure is more rule-like and transition-specific than globally learnable with the current features.

The strongest result in Notebook 16 remains:

uncertainty-gated t−1 = 0.5406 accuracy, with stable threshold 0.60 across all five folds.

And we also have strong structural intervention rules, especially:

TOOL_CALL→TOOL_CALL + workflow→tool_use: net +10
TOOL_CALL→ASSISTANT + workflow→grounding: net +10
TOOL_CALL→TOOL_CALL + workflow→grounding: net +5
ASSISTANT→TOOL_CALL + workflow→constraint: net +4

The next experiment I would run is a cross-fitted structural intervention policy: learn from each training fold which transition×prediction moves have positive utility, then only accept those moves on the held-out fold. This directly tests whether the rule structure generalizes without fitting another high-variance classifier.

In [92]:
# ============================================================
# 60. Cross-fitted structural intervention policy
# ============================================================

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)

# Global base / trajectory predictions
base_pred = remove_t1_pred.copy()
traj_pred = full_pred.copy()

base_correct = (
    base_pred == y_train_canonical
)

# Structural key for every row
semantic_family_all = decode_predictions(
    base_pred
)

trajectory_family_all = decode_predictions(
    traj_pred
)

structure_key_all = np.array([
    (
        transition_structure_df.loc[i, "transition_type"],
        semantic_family_all[i],
        trajectory_family_all[i],
    )
    for i in range(len(y_train_canonical))
], dtype=object)

# Only rows where trajectory changes the prediction
changed_mask = (
    base_pred != traj_pred
)

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=123,
)

crossfit_structural_pred = (
    base_pred.copy()
)

policy_rows = []

for fold, (tr_idx, va_idx) in enumerate(
    cv.split(
        np.zeros((len(y_train_canonical), 1)),
        y_train_canonical,
        groups=groups_train,
    ),
    start=1,
):

    # ---------------------------------------------
    # Learn structural utility from TRAIN fold
    # ---------------------------------------------

    train_changed = (
        changed_mask[tr_idx]
    )

    train_rows = tr_idx[
        train_changed
    ]

    train_policy = {}

    # collect all unique structural keys
    unique_keys = set(
        tuple(x)
        for x in structure_key_all[
            train_rows
        ]
    )

    for key in unique_keys:

        key_mask = np.array([
            tuple(structure_key_all[i]) == key
            for i in train_rows
        ])

        key_idx = train_rows[
            key_mask
        ]

        sem_correct = (
            base_pred[key_idx]
            == y_train_canonical[key_idx]
        )

        traj_correct = (
            traj_pred[key_idx]
            == y_train_canonical[key_idx]
        )

        rescues = int(
            (
                (~sem_correct)
                &
                traj_correct
            ).sum()
        )

        breaks = int(
            (
                sem_correct
                &
                (~traj_correct)
            ).sum()
        )

        support = len(key_idx)

        net = rescues - breaks

        # conservative structural policy:
        # require positive net and minimum support
        accept = (
            support >= 5
            and net > 0
        )

        train_policy[key] = accept

        policy_rows.append({
            "fold": fold,
            "transition_type": key[0],
            "semantic_family": key[1],
            "trajectory_family": key[2],
            "train_support": support,
            "train_rescues": rescues,
            "train_breaks": breaks,
            "train_net": net,
            "accept": accept,
        })

    # ---------------------------------------------
    # Apply to held-out fold
    # ---------------------------------------------

    for i in va_idx:

        if not changed_mask[i]:
            continue

        key = tuple(
            structure_key_all[i]
        )

        if train_policy.get(
            key,
            False,
        ):
            crossfit_structural_pred[i] = (
                traj_pred[i]
            )
        else:
            crossfit_structural_pred[i] = (
                base_pred[i]
            )

print("Done")

Done


In [93]:
# ============================================================
# 61. Evaluate cross-fitted structural policy
# ============================================================

def eval_pred(pred):
    return {
        "accuracy":
            accuracy_score(
                y_train_canonical,
                pred,
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_train_canonical,
                pred,
            ),

        "macro_f1":
            f1_score(
                y_train_canonical,
                pred,
                average="macro",
                zero_division=0,
            ),

        "weighted_f1":
            f1_score(
                y_train_canonical,
                pred,
                average="weighted",
                zero_division=0,
            ),
    }


print(
    "No t1:",
    eval_pred(
        base_pred
    )
)

print(
    "Full t1:",
    eval_pred(
        traj_pred
    )
)

print(
    "Structural crossfit:",
    eval_pred(
        crossfit_structural_pred
    )
)

struct_correct = (
    crossfit_structural_pred
    == y_train_canonical
)

rescues = (
    (~base_correct)
    &
    struct_correct
)

breaks = (
    base_correct
    &
    (~struct_correct)
)

print(
    "\nRescues:",
    rescues.sum()
)

print(
    "Breaks:",
    breaks.sum()
)

print(
    "Net:",
    rescues.sum()
    -
    breaks.sum()
)

No t1: {'accuracy': 0.5151108126259234, 'balanced_accuracy': 0.43550722847563206, 'macro_f1': 0.45483336181761536, 'weighted_f1': 0.4969282458539297}
Full t1: {'accuracy': 0.5379449294828744, 'balanced_accuracy': 0.49832469775945765, 'macro_f1': 0.5104901329429692, 'weighted_f1': 0.5351506945358958}
Structural crossfit: {'accuracy': 0.5319006044325051, 'balanced_accuracy': 0.47397891847038115, 'macro_f1': 0.49003858056402533, 'weighted_f1': 0.5260831340287168}

Rescues: 62
Breaks: 37
Net: 25


In [94]:
# ============================================================
# 62. Structural policy stability
# ============================================================

policy_df = pd.DataFrame(
    policy_rows
)

display(
    policy_df.sort_values(
        [
            "fold",
            "train_net",
        ],
        ascending=[
            True,
            False,
        ],
    )
)

policy_stability = (
    policy_df
    .groupby([
        "transition_type",
        "semantic_family",
        "trajectory_family",
    ])
    .agg(
        folds=("fold", "count"),
        mean_support=(
            "train_support",
            "mean",
        ),
        mean_net=(
            "train_net",
            "mean",
        ),
        accept_rate=(
            "accept",
            "mean",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "accept_rate",
            "mean_net",
        ],
        ascending=False,
    )
)

display(
    policy_stability.round(4)
)

,fold,transition_type,semantic_family,trajectory_family,train_support,train_rescues,train_breaks,train_net,accept
11,1,TOOL_CALL->ASSISTANT,workflow_error,grounding_state_error,14,9,3,6,True
12,1,TOOL_CALL->TOOL_CALL,workflow_error,tool_use_error,29,16,11,5,True
16,1,TOOL_CALL->TOOL_CALL,workflow_error,grounding_state_error,11,7,2,5,True
0,1,TOOL_CALL->ASSISTANT,constraint_error,grounding_state_error,6,3,1,2,True
13,1,TOOL_CALL->TOOL_CALL,constraint_error,grounding_state_error,2,2,0,2,False
...,...,...,...,...,...,...,...,...,...
95,5,ASSISTANT->TOOL_CALL,workflow_error,tool_use_error,17,7,7,0,False
96,5,ASSISTANT->ASSISTANT,reasoning_value_error,grounding_state_error,1,0,0,0,False
86,5,ASSISTANT->ASSISTANT,constraint_error,grounding_state_error,6,1,3,-2,False
92,5,ASSISTANT->ASSISTANT,workflow_error,grounding_state_error,7,1,3,-2,False


,transition_type,semantic_family,trajectory_family,folds,mean_support,mean_net,accept_rate
14,TOOL_CALL->ASSISTANT,workflow_error,grounding_state_error,5,15.2,8.0,1.0
20,TOOL_CALL->TOOL_CALL,workflow_error,tool_use_error,5,31.2,8.0,1.0
19,TOOL_CALL->TOOL_CALL,workflow_error,grounding_state_error,5,9.6,4.0,1.0
8,ASSISTANT->TOOL_CALL,workflow_error,constraint_error,5,12.0,3.2,1.0
11,TOOL_CALL->ASSISTANT,constraint_error,grounding_state_error,5,6.4,3.2,1.0
3,ASSISTANT->ASSISTANT,workflow_error,constraint_error,5,7.2,0.8,0.8
10,ASSISTANT->TOOL_CALL,workflow_error,tool_use_error,5,14.4,0.8,0.6
16,TOOL_CALL->ASSISTANT,workflow_error,tool_use_error,5,7.2,0.8,0.6
13,TOOL_CALL->ASSISTANT,workflow_error,constraint_error,5,12.0,-1.6,0.2
17,TOOL_CALL->TOOL_CALL,constraint_error,grounding_state_error,5,2.4,2.4,0.0


In [95]:
# ============================================================
# 63. Structural policy vs uncertainty gate overlap
# ============================================================

structural_changed = (
    crossfit_structural_pred
    != remove_t1_pred
)

uncertainty_changed = (
    crossfit_uncertainty_pred
    != remove_t1_pred
)

both = (
    structural_changed
    &
    uncertainty_changed
)

structural_only = (
    structural_changed
    &
    (~uncertainty_changed)
)

uncertainty_only = (
    uncertainty_changed
    &
    (~structural_changed)
)

neither = (
    (~structural_changed)
    &
    (~uncertainty_changed)
)

rows = []

for name, mask in {
    "both": both,
    "structural_only": structural_only,
    "uncertainty_only": uncertainty_only,
    "neither": neither,
}.items():

    n = int(mask.sum())

    if n == 0:
        continue

    rows.append({
        "subset": name,
        "support": n,

        "base_accuracy":
            (
                remove_t1_pred[mask]
                == y_train_canonical[mask]
            ).mean(),

        "structural_accuracy":
            (
                crossfit_structural_pred[mask]
                == y_train_canonical[mask]
            ).mean(),

        "uncertainty_accuracy":
            (
                crossfit_uncertainty_pred[mask]
                == y_train_canonical[mask]
            ).mean(),
    })

overlap_df = pd.DataFrame(rows)

display(
    overlap_df.round(4)
)

print("\nPolicy coverage")
print(
    "Structural:",
    structural_changed.sum()
)

print(
    "Uncertainty:",
    uncertainty_changed.sum()
)

print(
    "Both:",
    both.sum()
)

print(
    "Structural only:",
    structural_only.sum()
)

print(
    "Uncertainty only:",
    uncertainty_only.sum()
)

,subset,support,base_accuracy,structural_accuracy,uncertainty_accuracy
0,both,118,0.2966,0.5254,0.5254
1,structural_only,5,0.4000,0.0000,0.4000
2,uncertainty_only,37,0.2432,0.2432,0.5405
3,neither,1329,0.5425,0.5425,0.5425



Policy coverage
Structural: 123
Uncertainty: 155
Both: 118
Structural only: 5
Uncertainty only: 37


In [96]:
# ============================================================
# 64. Separate history vs no-history evaluation
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)

# ------------------------------------------------------------
# 1. Identify history availability
# ------------------------------------------------------------

# Prefer the explicit dataset field.
has_history = train_targets["has_history"].astype(bool).to_numpy()

print("Total rows:", len(has_history))
print("Has history:", has_history.sum())
print("No history:", (~has_history).sum())

# Sanity check against transition labels
if "transition_type" in transition_structure_df.columns:
    nohist_transition = (
        transition_structure_df["transition_type"]
        .astype(str)
        .str.startswith("NO_HISTORY")
        .to_numpy()
    )

    print(
        "\nNO_HISTORY transition rows:",
        nohist_transition.sum()
    )

    print(
        "Masks agree:",
        np.array_equal(~has_history, nohist_transition)
    )


# ------------------------------------------------------------
# 2. Metric helper
# ------------------------------------------------------------

def evaluate_subset(y_true, y_pred, mask):
    yt = np.asarray(y_true)[mask]
    yp = np.asarray(y_pred)[mask]

    return {
        "support": int(mask.sum()),
        "accuracy": accuracy_score(yt, yp),
        "balanced_accuracy": balanced_accuracy_score(yt, yp),
        "macro_f1": f1_score(
            yt,
            yp,
            average="macro",
            zero_division=0,
        ),
        "weighted_f1": f1_score(
            yt,
            yp,
            average="weighted",
            zero_division=0,
        ),
    }


# ------------------------------------------------------------
# 3. Compare all models
# ------------------------------------------------------------

models = {
    "no_t1": remove_t1_pred,
    "full_t1": full_pred,
    "structural_crossfit": crossfit_structural_pred,
    "uncertainty_crossfit": crossfit_uncertainty_pred,
}

rows = []

for subset_name, mask in {
    "ALL": np.ones(len(has_history), dtype=bool),
    "HAS_HISTORY": has_history,
    "NO_HISTORY": ~has_history,
}.items():

    for model_name, pred in models.items():

        result = evaluate_subset(
            y_train_canonical,
            pred,
            mask,
        )

        rows.append({
            "subset": subset_name,
            "model": model_name,
            **result,
        })

history_eval_df = pd.DataFrame(rows)

display(
    history_eval_df.round(4)
)


# ------------------------------------------------------------
# 4. More useful: deltas relative to no-t1
# ------------------------------------------------------------

metric_cols = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
]

delta_rows = []

for subset_name in history_eval_df["subset"].unique():

    sub = history_eval_df[
        history_eval_df["subset"] == subset_name
    ].set_index("model")

    base = sub.loc["no_t1"]

    for model_name in [
        "full_t1",
        "structural_crossfit",
        "uncertainty_crossfit",
    ]:

        row = {
            "subset": subset_name,
            "model": model_name,
            "support": int(sub.loc[model_name, "support"]),
        }

        for metric in metric_cols:
            row[f"delta_{metric}"] = (
                sub.loc[model_name, metric]
                - base[metric]
            )

        delta_rows.append(row)

history_delta_df = pd.DataFrame(delta_rows)

display(
    history_delta_df.round(4)
)


# ------------------------------------------------------------
# 5. Check whether trajectory policies ever intervene when
#    history is absent
# ------------------------------------------------------------

structural_changed = (
    np.asarray(crossfit_structural_pred)
    != np.asarray(remove_t1_pred)
)

uncertainty_changed = (
    np.asarray(crossfit_uncertainty_pred)
    != np.asarray(remove_t1_pred)
)

intervention_check = pd.DataFrame({
    "policy": [
        "structural",
        "uncertainty",
    ],
    "all_interventions": [
        structural_changed.sum(),
        uncertainty_changed.sum(),
    ],
    "history_interventions": [
        (structural_changed & has_history).sum(),
        (uncertainty_changed & has_history).sum(),
    ],
    "no_history_interventions": [
        (structural_changed & ~has_history).sum(),
        (uncertainty_changed & ~has_history).sum(),
    ],
})

display(intervention_check)

Total rows: 1489
Has history: 1433
No history: 56

NO_HISTORY transition rows: 56
Masks agree: True


,subset,model,support,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,ALL,no_t1,1489,0.5151,0.4355,0.4548,0.4969
1,ALL,full_t1,1489,0.5379,0.4983,0.5105,0.5352
2,ALL,structural_crossfit,1489,0.5319,0.4740,0.4900,0.5261
3,ALL,uncertainty_crossfit,1489,0.5406,0.4985,0.5105,0.5368
4,HAS_HISTORY,no_t1,1433,0.5136,0.4305,0.4519,0.4951
5,HAS_HISTORY,full_t1,1433,0.5373,0.4976,0.5116,0.5352
6,HAS_HISTORY,structural_crossfit,1433,0.5311,0.4716,0.4894,0.5257
7,HAS_HISTORY,uncertainty_crossfit,1433,0.5401,0.4977,0.5114,0.5368
8,NO_HISTORY,no_t1,56,0.5536,0.5440,0.4971,0.5153
9,NO_HISTORY,full_t1,56,0.5536,0.5440,0.4971,0.5153


,subset,model,support,delta_accuracy,delta_balanced_accuracy,delta_macro_f1,delta_weighted_f1
0,ALL,full_t1,1489,0.0228,0.0628,0.0557,0.0382
1,ALL,structural_crossfit,1489,0.0168,0.0385,0.0352,0.0292
2,ALL,uncertainty_crossfit,1489,0.0255,0.0630,0.0557,0.0398
3,HAS_HISTORY,full_t1,1433,0.0237,0.0671,0.0597,0.0401
4,HAS_HISTORY,structural_crossfit,1433,0.0174,0.0411,0.0376,0.0305
5,HAS_HISTORY,uncertainty_crossfit,1433,0.0265,0.0672,0.0595,0.0417
6,NO_HISTORY,full_t1,56,0.0000,0.0000,0.0000,0.0000
7,NO_HISTORY,structural_crossfit,56,0.0000,0.0000,0.0000,0.0000
8,NO_HISTORY,uncertainty_crossfit,56,0.0000,0.0000,0.0000,0.0000


,policy,all_interventions,history_interventions,no_history_interventions
0,structural,123,123,0
1,uncertainty,155,155,0


In [97]:
# ============================================================
# 65. What does uncertainty gating capture beyond structure?
# ============================================================

import numpy as np
import pandas as pd

structural_changed = (
    np.asarray(crossfit_structural_pred)
    != np.asarray(remove_t1_pred)
)

uncertainty_changed = (
    np.asarray(crossfit_uncertainty_pred)
    != np.asarray(remove_t1_pred)
)

uncertainty_only = (
    uncertainty_changed
    & ~structural_changed
)

idx = np.where(uncertainty_only)[0]

uo = pd.DataFrame({
    "idx": idx,

    "transition_type":
        transition_structure_df.iloc[idx]["transition_type"].to_numpy(),

    "true_family":
        np.asarray(y_train_canonical)[idx],

    "base_prediction":
        np.asarray(remove_t1_pred)[idx],

    "full_t1_prediction":
        np.asarray(full_pred)[idx],

    "uncertainty_prediction":
        np.asarray(crossfit_uncertainty_pred)[idx],

    "confidence":
        uncertainty_df.iloc[idx]["confidence"].to_numpy(),

    "margin":
        uncertainty_df.iloc[idx]["margin"].to_numpy(),

    "entropy":
        uncertainty_df.iloc[idx]["entropy"].to_numpy(),

    "t1_l1":
        distance_df.iloc[idx]["current_tminus1_l1"].to_numpy(),

    "t1_l2":
        distance_df.iloc[idx]["current_tminus1_l2"].to_numpy(),
})

# ------------------------------------------------------------
# Effect of uncertainty intervention
# ------------------------------------------------------------

uo["base_correct"] = (
    uo["base_prediction"] == uo["true_family"]
)

uo["uncertainty_correct"] = (
    uo["uncertainty_prediction"] == uo["true_family"]
)

uo["effect"] = np.select(
    [
        (~uo["base_correct"]) & uo["uncertainty_correct"],
        uo["base_correct"] & (~uo["uncertainty_correct"]),
    ],
    [
        "rescue",
        "break",
    ],
    default="wrong_to_wrong",
)

print("Uncertainty-only interventions:", len(uo))
print()

print("Effect distribution:")
print(uo["effect"].value_counts())

print("\nAccuracy:")
print(
    "Base:",
    round(uo["base_correct"].mean(), 4)
)
print(
    "Uncertainty:",
    round(uo["uncertainty_correct"].mean(), 4)
)


# ------------------------------------------------------------
# Prediction transitions
# ------------------------------------------------------------

uo["prediction_transition"] = (
    uo["base_prediction"].astype(str)
    + " -> "
    + uo["uncertainty_prediction"].astype(str)
)

transition_summary = (
    uo.groupby(
        [
            "transition_type",
            "prediction_transition",
        ]
    )
    .agg(
        support=("effect", "size"),
        rescues=("effect", lambda x: (x == "rescue").sum()),
        breaks=("effect", lambda x: (x == "break").sum()),
        wrong_to_wrong=(
            "effect",
            lambda x: (x == "wrong_to_wrong").sum()
        ),
        mean_confidence=("confidence", "mean"),
        mean_margin=("margin", "mean"),
        mean_entropy=("entropy", "mean"),
        mean_t1_l1=("t1_l1", "mean"),
        mean_t1_l2=("t1_l2", "mean"),
    )
    .reset_index()
)

transition_summary["net"] = (
    transition_summary["rescues"]
    - transition_summary["breaks"]
)

transition_summary["rescue_rate"] = (
    transition_summary["rescues"]
    / transition_summary["support"]
)

display(
    transition_summary
    .sort_values(
        ["net", "support"],
        ascending=False,
    )
    .round(4)
)


# ------------------------------------------------------------
# Aggregate just by event transition
# ------------------------------------------------------------

event_summary = (
    uo.groupby("transition_type")
    .agg(
        support=("effect", "size"),
        rescues=("effect", lambda x: (x == "rescue").sum()),
        breaks=("effect", lambda x: (x == "break").sum()),
        wrong_to_wrong=(
            "effect",
            lambda x: (x == "wrong_to_wrong").sum()
        ),
    )
    .reset_index()
)

event_summary["net"] = (
    event_summary["rescues"]
    - event_summary["breaks"]
)

event_summary["accuracy"] = (
    event_summary["rescues"]
    / event_summary["support"]
)

display(
    event_summary
    .sort_values("net", ascending=False)
    .round(4)
)


# ------------------------------------------------------------
# Inspect actual rows
# ------------------------------------------------------------

display(
    uo.sort_values(
        ["effect", "transition_type"]
    )
)

Uncertainty-only interventions: 37

Effect distribution:
effect
rescue            20
break              9
wrong_to_wrong     8
Name: count, dtype: int64

Accuracy:
Base: 0.2432
Uncertainty: 0.5405


,transition_type,prediction_transition,support,rescues,breaks,wrong_to_wrong,mean_confidence,mean_margin,mean_entropy,mean_t1_l1,mean_t1_l2,net,rescue_rate
5,TOOL_CALL->ASSISTANT,0 -> 2,3,3,0,0,0.3833,0.0475,1.2121,0.0435,1.0643,3,1.0000
9,TOOL_CALL->TOOL_CALL,1 -> 3,3,3,0,0,0.4731,0.0859,1.0709,0.0364,0.8931,3,1.0000
0,ASSISTANT->TOOL_CALL,0 -> 2,2,2,0,0,0.4114,0.1255,1.2542,0.0499,1.2282,2,1.0000
4,TOOL_CALL->ASSISTANT,0 -> 1,12,5,4,3,0.4764,0.0922,1.0800,0.0461,1.1356,1,0.4167
8,TOOL_CALL->TOOL_CALL,0 -> 1,2,1,0,1,0.3915,0.0385,1.2206,0.0438,1.0702,1,0.5000
3,ASSISTANT->TOOL_CALL,1 -> 3,1,1,0,0,0.3466,0.0826,1.3664,0.0482,1.2048,1,1.0000
6,TOOL_CALL->ASSISTANT,0 -> 4,1,1,0,0,0.3903,0.1100,1.3881,0.0473,1.1922,1,1.0000
7,TOOL_CALL->ASSISTANT,1 -> 4,1,1,0,0,0.3871,0.0701,1.3789,0.0471,1.1547,1,1.0000
2,ASSISTANT->TOOL_CALL,1 -> 2,1,0,0,1,0.3326,0.0197,1.3629,0.0546,1.3458,0,0.0000
1,ASSISTANT->TOOL_CALL,0 -> 3,11,3,5,3,0.4906,0.1606,1.1331,0.0485,1.1902,-2,0.2727


,transition_type,support,rescues,breaks,wrong_to_wrong,net,accuracy
1,TOOL_CALL->ASSISTANT,17,10,4,3,6,0.5882
2,TOOL_CALL->TOOL_CALL,5,4,0,1,4,0.8000
0,ASSISTANT->TOOL_CALL,15,6,5,4,1,0.4000


,idx,transition_type,true_family,base_prediction,full_t1_prediction,uncertainty_prediction,confidence,margin,entropy,t1_l1,t1_l2,base_correct,uncertainty_correct,effect,prediction_transition
14,1073,ASSISTANT->TOOL_CALL,0,0,3,3,0.363937,0.018675,1.329134,0.046764,1.153676,True,False,break,0 -> 3
15,1078,ASSISTANT->TOOL_CALL,0,0,3,3,0.598439,0.274585,0.938284,0.053133,1.335063,True,False,break,0 -> 3
17,1103,ASSISTANT->TOOL_CALL,0,0,3,3,0.346374,0.001126,1.280945,0.044486,1.089413,True,False,break,0 -> 3
31,1448,ASSISTANT->TOOL_CALL,0,0,3,3,0.527269,0.195054,1.129628,0.051027,1.249893,True,False,break,0 -> 3
34,1465,ASSISTANT->TOOL_CALL,0,0,3,3,0.549752,0.284867,1.175847,0.051706,1.259184,True,False,break,0 -> 3
3,550,TOOL_CALL->ASSISTANT,0,0,1,1,0.556727,0.125189,0.752749,0.045633,1.140325,True,False,break,0 -> 1
5,589,TOOL_CALL->ASSISTANT,0,0,1,1,0.470395,0.133300,1.194209,0.048977,1.234101,True,False,break,0 -> 1
6,593,TOOL_CALL->ASSISTANT,0,0,1,1,0.547560,0.106992,0.755541,0.040707,1.028922,True,False,break,0 -> 1
12,989,TOOL_CALL->ASSISTANT,0,0,1,1,0.392978,0.046764,1.291150,0.044168,1.082358,True,False,break,0 -> 1
1,120,ASSISTANT->TOOL_CALL,2,0,2,2,0.397831,0.103754,1.328236,0.052333,1.302068,False,True,rescue,0 -> 2


# `16_event_transition_failure_mechanisms.ipynb` — Full Notebook Summary and Conclusions

## Notebook objective

This notebook investigated **how event-transition structure changes the usefulness of trajectory context for failure classification**, and whether that structure can support a reliable routing mechanism.

The starting point was the observation from earlier notebooks that adding trajectory information improves the classifier overall, but not uniformly across examples or failure families. Notebook 16 therefore moved away from asking only:

> “Does trajectory context improve classification?”

and instead asked:

> **When does recent trajectory context help, when does it hurt, and what mechanism explains the difference?**

The notebook progressively examined:

1. whether `t−1` trajectory utility depends on event transition type,
2. whether semantic uncertainty can gate trajectory usage,
3. whether raw distance itself explains rescues,
4. whether trajectory effects correspond to specific class transitions,
5. whether specialized refinement experts are learnable,
6. whether a learned intervention verifier can distinguish rescues from breaks,
7. whether a simple structural intervention policy generalizes.

---

# 1. Transition structure of the dataset

The recent event structure was represented through transition types such as:

* `TOOL_CALL -> TOOL_CALL`
* `TOOL_CALL -> ASSISTANT`
* `ASSISTANT -> TOOL_CALL`
* `ASSISTANT -> ASSISTANT`
* `NO_HISTORY -> ASSISTANT`
* `NO_HISTORY -> TOOL_CALL`

The distribution showed that tool-mediated transitions dominate the dataset:

```text
TOOL_CALL -> TOOL_CALL     590
TOOL_CALL -> ASSISTANT     397
ASSISTANT -> ASSISTANT     226
ASSISTANT -> TOOL_CALL     220
NO_HISTORY -> ASSISTANT     34
NO_HISTORY -> TOOL_CALL     22
```

This immediately suggested that “one previous event” does not have a single semantic meaning. The same `t−1` distance can correspond to very different functional relations:

* tool action following another tool action,
* assistant response following a tool action,
* tool action following assistant reasoning,
* assistant response following another assistant response.

That became the core mechanism studied throughout the notebook.

---

# 2. `t−1` is globally important, but its utility is transition-dependent

Removing `t−1` caused a large drop:

| Representation |   Accuracy | Balanced Accuracy |   Macro-F1 |
| -------------- | ---------: | ----------------: | ---------: |
| without `t−1`  |     0.5151 |            0.4355 |     0.4548 |
| full `t−1`     | **0.5379** |        **0.4983** | **0.5105** |

So recent trajectory evidence is clearly useful.

However, transition-conditioned ablation showed that this utility is highly heterogeneous.

The strongest positive effects of `t−1` included:

* `TOOL_CALL -> TOOL_CALL`, `tool_use_error`: large positive recall contribution.
* `TOOL_CALL -> ASSISTANT`, `grounding_state_error`: very strong positive contribution.
* `ASSISTANT -> TOOL_CALL`, `constraint_error`: positive contribution.
* `ASSISTANT -> TOOL_CALL`, `tool_use_error`: positive contribution.

At the same time, `t−1` could be actively harmful for:

* workflow errors,
* especially in some assistant/tool transition configurations,
* and `ASSISTANT -> ASSISTANT` overall.

This invalidated the simple idea:

```python
use_t1 = event_has_history
```

and also made a hard role policy such as:

```python
TOOL_CALL->TOOL_CALL     True
TOOL_CALL->ASSISTANT     True
ASSISTANT->TOOL_CALL     True
ASSISTANT->ASSISTANT     False
```

too coarse.

---

# 3. Semantic uncertainty strongly controls trajectory utility

A major notebook result came from conditioning `t−1` utility on semantic confidence.

For low-confidence cases:

| Transition               | Net `t−1` utility |
| ------------------------ | ----------------: |
| `TOOL_CALL -> ASSISTANT` |               +16 |
| `TOOL_CALL -> TOOL_CALL` |               +13 |
| `ASSISTANT -> TOOL_CALL` |                +7 |

At high confidence, `t−1` often produced no prediction change at all.

The entropy analysis confirmed the same pattern from the opposite direction:

* high entropy / uncertain examples benefited more from trajectory evidence,
* especially in tool-mediated transitions,
* `ASSISTANT -> ASSISTANT` remained neutral or negative.

This produced the first useful gating mechanism:

> **Trajectory context is most valuable when the semantic model is uncertain and the current event is part of a tool-mediated transition.**

---

# 4. Uncertainty-gated trajectory model

A confidence threshold was tested for selectively applying the full `t−1` model.

The best region showed a broad plateau. Around confidence threshold `0.60`, the model achieved:

```text
accuracy            0.5406
balanced_accuracy   0.4985
macro_f1            0.5105
weighted_f1         0.5368

rescues             82
breaks              44
net                  +38
```

The important validation result was that **all five folds selected the same threshold: `0.60`**.

Cross-fitted result:

| Model                       |   Accuracy | Balanced Accuracy |   Macro-F1 |
| --------------------------- | ---------: | ----------------: | ---------: |
| no `t−1`                    |     0.5151 |            0.4355 |     0.4548 |
| full `t−1`                  |     0.5379 |            0.4983 |     0.5105 |
| **uncertainty-gated `t−1`** | **0.5406** |        **0.4985** | **0.5105** |

This became the strongest model in the notebook.

### Interpretation

The gate does not seem to identify a tiny precise subset where history is uniquely useful. Instead, semantic confidence identifies a **region where trajectory evidence is allowed to affect the classifier**.

Many examples above the threshold can receive trajectory features without changing the final class, explaining the wide performance plateau.

---

# 5. Raw `t−1` distance is not the failure detector

A critical negative result was that raw displacement magnitude does not explain the mechanism.

For `TOOL_CALL -> ASSISTANT`, grounding errors versus non-grounding examples had almost identical distances:

```text
L1 raw AUC ≈ 0.522
L2 raw AUC ≈ 0.519
```

Similarly, rescue versus break geometry was very similar within transitions.

Later, on the complete rescue/break intervention set:

```text
t1_l1 AUC ≈ 0.507
t1_l2 AUC ≈ 0.506
```

essentially chance.

Therefore:

> **“Large semantic displacement” is not equivalent to “failure.”**

Trajectory information is useful because it changes the classifier state in context, not because an absolute distance threshold identifies an error.

This was one of the most important mechanistic conclusions of the notebook.

---

# 6. Trajectory primarily refines `workflow_error`

One of the clearest structural discoveries was how predictions change when `t−1` is introduced.

There were:

```text
Changed predictions: 181
Moves into workflow: 0
Moves out of workflow: 159
```

Trajectory predictions among changed cases:

```text
grounding_state_error    69
tool_use_error           68
constraint_error         42
reasoning_value_error     2
```

Semantic/no-`t−1` predictions among changed cases:

```text
workflow_error           159
constraint_error          20
reasoning_value_error      1
tool_use_error             1
```

This means the trajectory model behaves overwhelmingly as:

> **a refinement mechanism that moves examples out of the generic `workflow_error` basin into more specific local failure classes.**

It practically never moves examples into workflow.

This changed the interpretation of trajectory context substantially.

Instead of being a second generic classifier, trajectory evidence behaves like a **local failure disambiguator**.

---

# 7. `workflow_error` is a catch-all semantic basin

Among the 873 examples predicted as `workflow_error` by the no-`t1` model:

```text
true workflow_error           483
constraint_error              146
tool_use_error                131
grounding_state_error         107
reasoning_value_error           6
```

So only about 55% of semantic workflow predictions are actually workflow errors.

The rest are hidden specific failure families.

This gives an important conceptual interpretation:

> **The semantic classifier collapses many ambiguous local failures into the broad workflow category.**

Trajectory evidence can sometimes resolve that ambiguity.

---

# 8. Stable transition-specific correction mechanisms

The prediction-change analysis revealed several strong, interpretable corrections.

The largest positive moves were:

### `TOOL_CALL -> TOOL_CALL`

```text
workflow_error -> tool_use_error
support 39
rescues 23
breaks 13
net +10
```

### `TOOL_CALL -> ASSISTANT`

```text
workflow_error -> grounding_state_error
support 19
rescues 13
breaks 3
net +10
```

### `TOOL_CALL -> TOOL_CALL`

```text
workflow_error -> grounding_state_error
support 12
rescues 7
breaks 2
net +5
```

### `ASSISTANT -> TOOL_CALL`

```text
workflow_error -> constraint_error
support 15
rescues 8
breaks 4
net +4
```

There were also stable harmful moves:

### `TOOL_CALL -> ASSISTANT`

```text
workflow_error -> constraint_error
net -2
```

### `ASSISTANT -> TOOL_CALL`

```text
workflow_error -> grounding_state_error
net -2
```

### `ASSISTANT -> ASSISTANT`

```text
workflow_error -> grounding_state_error
net -2
```

This strongly supports:

> **The meaning of a trajectory-induced class change depends on the event relation in which it occurs.**

---

# 9. Specialized refinement tasks

The notebook then reframed three important corrections as binary refinement tasks.

### Tool→Tool: is semantic workflow actually tool-use?

```text
support       407
positives      72
prevalence   0.1769
```

### Tool→Assistant: is semantic workflow actually grounding?

```text
support       190
positives      39
prevalence   0.2053
```

### Assistant→Tool: is semantic workflow actually constraint?

```text
support       139
positives      30
prevalence   0.2158
```

These were learnable ranking problems.

Cross-fitted performance:

| Task                        |     PR-AUC |    ROC-AUC |
| --------------------------- | ---------: | ---------: |
| Tool→Tool → tool-use        |     0.4734 | **0.8243** |
| Tool→Assistant → grounding  | **0.5053** |     0.7156 |
| Assistant→Tool → constraint | **0.5135** |     0.7177 |

These were far above their prevalence baselines.

So there is clearly enough information to rank refinement opportunities.

---

# 10. Feature-ablation result: different mechanisms use different evidence

The specialist feature ablation was especially informative.

## Tool→Tool / tool-use

| Features            |     PR-AUC |    ROC-AUC |
| ------------------- | ---------: | ---------: |
| semantic state      |     0.4593 | **0.8362** |
| distance only       |     0.2590 |     0.6417 |
| semantic + distance | **0.4734** |     0.8243 |
| full router         |     0.4541 |     0.8224 |

Interpretation:

> Tool-use refinement is already strongly encoded in semantic uncertainty/class competition. Distance adds only a small gain.

---

## Tool→Assistant / grounding

| Features            |     PR-AUC |    ROC-AUC |
| ------------------- | ---------: | ---------: |
| semantic state      |     0.5053 | **0.7156** |
| distance only       |     0.2541 |     0.5444 |
| semantic + distance | **0.5075** |     0.6972 |
| full router         |     0.4898 |     0.6962 |

Interpretation:

> Generic trajectory distance adds almost nothing for grounding. The current semantic state already contains most of the useful signal.

This strongly suggests that grounding needs a **better relation-specific representation**, probably tool-result ↔ assistant-response consistency.

---

## Assistant→Tool / constraint

| Features            |     PR-AUC |    ROC-AUC |
| ------------------- | ---------: | ---------: |
| semantic state      |     0.4471 |     0.7015 |
| distance only       |     0.3528 |     0.6480 |
| semantic + distance |     0.4100 |     0.7107 |
| **full router**     | **0.5135** | **0.7177** |

Interpretation:

> Constraint refinement is the clearest branch where the way trajectory changes the model probabilities matters more than raw distance.

This looks like a genuine meta-routing problem.

---

# 11. Independent specialist correction models did not beat full trajectory

The three specialists were then used directly to override workflow predictions.

Cross-fitted individual results showed modest or weak gains.

Combined specialists:

```text
accuracy            0.5198
balanced_accuracy   0.4768
macro_f1            0.4855
weighted_f1         0.5184

rescues             78
breaks              71
net                  +7
```

This was better than no `t−1`, but substantially worse than both:

* full `t−1`: `0.5379`
* uncertainty-gated `t−1`: `0.5406`

Therefore:

> **Specialists are useful diagnostically, but they should not independently replace the trajectory model.**

Their better role is to describe or verify proposed trajectory refinements.

---

# 12. Intervention view of trajectory

The notebook then reframed trajectory as an **intervention on the semantic prediction**.

Correct class mapping was recovered as:

```text
0 -> workflow_error
1 -> constraint_error
2 -> tool_use_error
3 -> grounding_state_error
4 -> reasoning_value_error
```

This fixed an important class-index mismatch encountered during analysis.

Across the 181 trajectory-induced prediction changes:

```text
rescues           87
breaks            53
wrong_to_wrong    41
```

This produced a clean intervention question:

> When trajectory proposes a different class, should we accept that intervention?

---

# 13. What distinguishes rescues from breaks?

The strongest univariate rescue-vs-break features were not distances.

| Feature                         |        AUC |
| ------------------------------- | ---------: |
| confidence gain                 | **0.6255** |
| trajectory margin               | **0.6183** |
| margin gain                     | **0.6137** |
| entropy drop                    |     0.5964 |
| trajectory confidence           |     0.5702 |
| proposed-class probability gain |     0.5689 |
| t1 L1                           |     0.5068 |
| t1 L2                           |     0.5057 |

This is a major mechanistic finding.

Rescues tend to occur when adding trajectory evidence:

* increases classifier confidence,
* increases the margin between the top two classes,
* produces a more decisive alternative prediction.

Breaks are more associated with trajectory changes that do **not** strengthen decision quality.

Therefore:

> **Trajectory reliability is better characterized by the change in classifier state than by raw geometric displacement.**

---

# 14. Generic learned rescue-vs-break verifier failed

A logistic regression verifier was trained on the 140 clean rescue/break interventions.

Results:

```text
PR-AUC  = 0.6239
ROC-AUC = 0.5133
```

Because rescue prevalence was already:

```text
0.6214
```

the PR-AUC provided essentially no improvement over baseline prevalence.

ROC-AUC near 0.51 confirmed that the model had no useful ranking ability.

Thresholding also failed to beat the trajectory baselines.

So:

> **A generic learned verifier over current probability-change + structural features does not generalize.**

This was an important negative result and prevented unnecessary further router tuning.

---

# 15. Cross-fitted structural intervention policy

A simpler structural policy was then tested.

For each training fold, the model learned which combinations of:

```text
event transition
+
semantic predicted family
+
trajectory proposed family
```

had positive net utility.

It accepted only those moves in the held-out fold.

Result:

| Model                   |   Accuracy | Balanced Accuracy |   Macro-F1 |
| ----------------------- | ---------: | ----------------: | ---------: |
| no `t−1`                |     0.5151 |            0.4355 |     0.4548 |
| structural cross-fit    | **0.5319** |        **0.4740** | **0.4900** |
| full `t−1`              |     0.5379 |            0.4983 |     0.5105 |
| uncertainty-gated `t−1` | **0.5406** |        **0.4985** | **0.5105** |

Structural policy:

```text
rescues 62
breaks  37
net     +25
```

So structural routing clearly generalizes and is meaningful, but it is too conservative to match the full trajectory model.

---

# 16. Structural policy stability

Several structural moves were selected in **all five folds**:

| Transition     | Semantic → trajectory  | Mean net | Accept rate |
| -------------- | ---------------------- | -------: | ----------: |
| Tool→Assistant | workflow → grounding   | **+8.0** |         1.0 |
| Tool→Tool      | workflow → tool-use    | **+8.0** |         1.0 |
| Tool→Tool      | workflow → grounding   | **+4.0** |         1.0 |
| Assistant→Tool | workflow → constraint  | **+3.2** |         1.0 |
| Tool→Assistant | constraint → grounding | **+3.2** |         1.0 |

Stable harmful structures included:

* Assistant→Assistant constraint→grounding
* Assistant→Assistant workflow→grounding
* Assistant→Tool workflow→grounding

This provides strong evidence that **transition-conditioned failure mechanisms are reproducible rather than accidental**.

---

# Final conclusions

## 1. Recent trajectory context is genuinely useful

Removing `t−1` causes a large degradation:

```text
accuracy:   0.5379 → 0.5151
macro-F1:   0.5105 → 0.4548
```

So recent history is one of the strongest non-semantic signals discovered so far.

---

## 2. But trajectory utility is not uniform

Its usefulness depends strongly on:

* semantic uncertainty,
* event transition,
* semantic predicted class,
* trajectory proposed class.

Therefore trajectory should not simply be concatenated globally and trusted everywhere.

---

## 3. The strongest practical model is uncertainty-gated trajectory

Current best Notebook 16 result:

```text
accuracy            0.5406
balanced_accuracy   0.4985
macro_f1            0.5105
weighted_f1         0.5368
```

The threshold `0.60` was selected consistently across all folds.

This is the most robust routing result from the notebook.

---

## 4. Raw trajectory distance is not the mechanism

L1/L2 displacement alone is nearly useless for telling rescues from breaks.

The useful signal comes from **how trajectory changes the classifier's belief state**, especially confidence and margin.

So additional engineering of generic L1/L2 transformations is unlikely to produce major gains.

---

## 5. `workflow_error` is a semantic ambiguity basin

The no-history semantic model heavily overpredicts workflow errors.

Trajectory primarily acts by moving these examples into more specific classes.

This is not symmetric routing.

It is better described as:

> **generic workflow diagnosis → local mechanism refinement**

---

## 6. Failure refinements are relation-specific

The strongest corrections have intuitive event semantics:

```text
TOOL_CALL → TOOL_CALL
workflow → tool_use

TOOL_CALL → ASSISTANT
workflow → grounding

ASSISTANT → TOOL_CALL
workflow → constraint
```

This provides direct empirical justification for **relation-aware reliability modeling**.

---

## 7. Different failure mechanisms require different representations

The specialist ablations showed:

* tool-use refinement is mostly semantic-state driven,
* grounding is not captured well by generic t−1 distance,
* constraint refinement benefits from model-disagreement / probability-change information.

Therefore a single shared trajectory expert is unlikely to be optimal.

---

## 8. A generic learned router is not currently justified

Neither:

* independent specialists,
* nor a generic rescue/break verifier,

beat the uncertainty-gated trajectory model.

The bottleneck is therefore **not router complexity**.

The bottleneck is increasingly **representation quality**.

---

# Main research conclusion

The notebook supports the following view of the reliability architecture:

```text
                    CURRENT EVENT
                         │
                         ▼
                SEMANTIC CLASSIFIER
                         │
                 semantic belief state
                         │
                 uncertainty / ambiguity
                         │
                         ▼
                  TRAJECTORY EVIDENCE
                         │
             proposes local refinement
                         │
       ┌─────────────────┼──────────────────┐
       │                 │                  │
 TOOL→TOOL          TOOL→ASSISTANT    ASSISTANT→TOOL
       │                 │                  │
 tool-use           grounding          constraint /
 evidence            evidence          action validity
       │                 │                  │
       └─────────────────┼──────────────────┘
                         ▼
                  FINAL RELIABILITY
                     PREDICTION
```

The crucial shift is:

> **History should be represented through the functional relation between events, not merely temporal position.**

---

# Recommended next notebook

## `17_relation_specific_evidence_representations.ipynb`

The next research should stop optimizing generic trajectory routing and instead build relation-specific evidence.

The most important research questions are:

1. **Tool result → assistant response:** can explicit consistency features detect grounding/hallucination failures better than generic `t−1` distance?
2. **Assistant intent → tool call:** can relation features identify constraint or tool-use violations?
3. **Tool call → next tool call:** can tool-transition representations distinguish legitimate workflow progression from tool-use errors?
4. **User constraint → later action:** can long-range constraint compliance features recover constraint errors that local distances miss?
5. Can these relation-specific features improve the existing **0.5406 uncertainty-gated trajectory baseline** without increasing breaks?

The most important final takeaway from Notebook 16 is:

> **Trajectory context works, but it works because event relations carry different meanings. Generic distance features expose some of this signal indirectly; the next step is to model those relations directly.**
